# Statistical significance analyses

Pooled and fold-averaged paired student-cluster analyses for both prediction tasks. Run the required section after updating its OOF paths.


In [ ]:
# First Attempt Pooled Statistical Significance

"""
First Attempt statistical significance analysis: best fine-grained vs best semantic DLKT configuration.

Selection basis
---------------
The compared configurations are selected from the first-attempt results table reported in the corresponding results table.
Fine-grained KC family:
    actionableelementid, itemid, itemsetid, exerciseid
Semantic KC family:
    propertyexercisetype, linguisticconstructs, propertyid

Exercise ID is treated as fine-grained because it is an identifier-level KC representation.
If you later decide to exclude Exercise ID from the fine-grained family, only the Accuracy
winner changes in the supplied table (from DKTForget+Exercise ID to DKTForget+Item ID).

Important statistical change from the old script
-------------------------------------------------
OOF rows are repeated observations nested within students. Therefore, this script does NOT
use row-level DeLong/McNemar/Wilcoxon as the primary significance tests. Instead it uses:

  1) paired STUDENT-CLUSTER permutation tests for two-sided p-values; and
  2) paired STUDENT-CLUSTER bootstrap for 95% confidence intervals.

For the bootstrap, students are resampled with replacement within each outer fold. All rows
belonging to a sampled student are kept together. This preserves within-student dependence
and the outer-fold structure.

All comparisons are paired on common row_id values only. student_id, y_true, and fold are
explicitly checked for agreement across the two OOF files.

The script also applies Holm correction across the family of comparisons.

Expected OOF layout from the evaluation notebooks
--------------------------------------------
    AKT_FA/oof_AKT_<kc_name>_all.csv
    SAKT_FA/oof_SAKT_<kc_name>_all.csv
    DKVMN_FA/oof_DKVMN_<kc_name>_all.csv
    SKVMN_FA/oof_SKVMN_<kc_name>_all.csv
    DKT_FA/oof_DKT_<kc_name>_all.csv
    DKTForget_FA/oof_DKTForget_<kc_name>_all.csv

Required OOF columns:
    row_id, student_id, fold, y_true, y_pred, model_name, kc_name
"""

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
)
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Run this script from the directory containing AKT_FA, DKTForget_FA, etc.,
# or change OOF_ROOT to the parent directory containing those folders.
OOF_ROOT = Path(".")

# Match the KT wrappers: probability must be STRICTLY greater than 0.5.
THRESHOLD = 0.5

# Student-cluster resampling settings.
# For a quick smoke test, temporarily reduce these to e.g. 200.
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 10000
RANDOM_SEED = 42
ALPHA = 0.05

# Exercise ID is included in the fine-grained family.
FINE_GRAINED_KCS = {
    "actionableelementid",
    "itemid",
    "itemsetid",
    "exerciseid",
}

SEMANTIC_KCS = {
    "propertyexercisetype",
    "linguisticconstructs",
    "propertyid",
}


def oof_path(folder: str, model: str, kc: str) -> Path:
    """Build the OOF path, using student-ID-fixed files for AKT."""
    if model == "AKT":
        return OOF_ROOT / folder / f"oof_{model}_{kc}_all_with_student_id.csv"

    return OOF_ROOT / folder / f"oof_{model}_{kc}_all.csv"


# Best fine-grained and semantic configuration for each metric, extracted from
# the first-attempt table reported in the corresponding results table.
#
# NOTE ON MAE:
# The unrounded fold-average values resolve the rounded tie:
#   Linguistic Constructs = 0.262597 ± 0.004783
#   Property + Exercise Type = 0.263345 ± 0.001364
# Lower MAE is better, so only Linguistic Constructs is retained as the
# best semantic configuration for the MAE significance comparison.
# Exactly one fine-grained-vs-semantic comparison per metric (7 total).
# COMPARISONS = [
#     {
#         "comparison_id": "AUC",
#         "metric": "AUC",
#         "fine_name": "DKT+Forget + Actionable Element ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "actionableelementid"),
#         "fine_table_value": "0.824673 ± 0.002840",
#         "semantic_name": "DKT+Forget + Property + Exercise Type",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "propertyexercisetype"),
#         "semantic_table_value": "0.821482 ± 0.005384",
#     },
#     {
#         "comparison_id": "Accuracy",
#         "metric": "Accuracy",
#         "fine_name": "DKT+Forget + Exercise ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "exerciseid"),
#         "fine_table_value": "0.792137 ± 0.008172",
#         "semantic_name": "DKT+Forget + Property + Exercise Type",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "propertyexercisetype"),
#         "semantic_table_value": "0.783335 ± 0.002057",
#     },
#     {
#         "comparison_id": "RMSE",
#         "metric": "RMSE",
#         "fine_name": "DKT+Forget + Item ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "itemid"),
#         "fine_table_value": "0.387209 ± 0.003727",
#         "semantic_name": "DKT+Forget + Property + Exercise Type",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "propertyexercisetype"),
#         "semantic_table_value": "0.388825 ± 0.002148",
#     },
#     {
#         "comparison_id": "MAE",
#         "metric": "MAE",
#         "fine_name": "DKT+Forget + Actionable Element ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "actionableelementid"),
#         "fine_table_value": "0.255216 ± 0.002966",
#         "semantic_name": "DKT+Forget + Linguistic Constructs",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "linguisticconstructs"),
#         "semantic_table_value": "0.262597 ± 0.004783",
#     },
#     {
#         "comparison_id": "Precision",
#         "metric": "Precision",
#         "fine_name": "DKT+Forget + Actionable Element ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "actionableelementid"),
#         "fine_table_value": "0.830726 ± 0.008355",
#         "semantic_name": "DKT+Forget + Property + Exercise Type",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "propertyexercisetype"),
#         "semantic_table_value": "0.822530 ± 0.007814",
#     },
#     {
#         "comparison_id": "Recall",
#         "metric": "Recall",
#         "fine_name": "AKT + Item Set ID",
#         "fine_file": oof_path("AKT_FA", "AKT", "itemsetid"),
#         "fine_table_value": "0.919 ± 0.008",
#         "semantic_name": "DKVMN + Property ID",
#         "semantic_file": oof_path("DKVMN_FA", "DKVMN", "propertyid"),
#         "semantic_table_value": "0.919 ± 0.004",
#     },
#     {
#         "comparison_id": "F1",
#         "metric": "F1",
#         "fine_name": "DKT+Forget + Item ID",
#         "fine_file": oof_path("DKTForget_FA", "DKTForget", "itemid"),
#         "fine_table_value": "0.859073 ± 0.003567",
#         "semantic_name": "DKT+Forget + Property + Exercise Type",
#         "semantic_file": oof_path("DKTForget_FA", "DKTForget", "propertyexercisetype"),
#         "semantic_table_value": "0.855106 ± 0.001185",
#     },
# ]

COMPARISONS = [
    {
        "comparison_id": "AUC",
        "metric": "AUC",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.825 ± 0.003",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.821 ± 0.005",
    },
    {
        "comparison_id": "Accuracy",
        "metric": "Accuracy",
        "fine_name": "DKT+Forget + Exercise ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "exerciseid"
        ),
        "fine_table_value": "0.792 ± 0.008",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.783 ± 0.002",
    },
    {
        "comparison_id": "RMSE",
        "metric": "RMSE",
        "fine_name": "DKT+Forget + Item ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "itemid"
        ),
        "fine_table_value": "0.387 ± 0.004",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.389 ± 0.002",
    },
    {
        "comparison_id": "MAE",
        "metric": "MAE",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.255 ± 0.003",
        "semantic_name": "DKT+Forget + Linguistic Constructs",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "linguisticconstructs"
        ),
        "semantic_table_value": "0.263 ± 0.005",
    },
    {
        "comparison_id": "Precision",
        "metric": "Precision",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.831 ± 0.008",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.823 ± 0.008",
    },
    {
        "comparison_id": "Recall",
        "metric": "Recall",
        "fine_name": "AKT + Item Set ID",
        "fine_file": oof_path(
            "AKT_FA", "AKT", "itemsetid"
        ),
        "fine_table_value": "0.919 ± 0.008",
        "semantic_name": "DKVMN + Property ID",
        "semantic_file": oof_path(
            "DKVMN_FA", "DKVMN", "propertyid"
        ),
        "semantic_table_value": "0.919 ± 0.004",
    },
    {
        "comparison_id": "F1",
        "metric": "F1",
        "fine_name": "DKT+Forget + Item ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "itemid"
        ),
        "fine_table_value": "0.859 ± 0.004",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.855 ± 0.001",
    },
]


# ============================================================
# 2. METRICS
# ============================================================

def rmse_score(y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_prob) ** 2)))


def metric_value(metric: str, y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    if metric == "AUC":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))

    # Match the model wrappers exactly: p > 0.5 -> 1, otherwise 0.
    y_hat = (y_prob > THRESHOLD).astype(int)

    if metric == "Accuracy":
        return float(accuracy_score(y_true, y_hat))
    if metric == "RMSE":
        return rmse_score(y_true, y_prob)
    if metric == "MAE":
        return float(mean_absolute_error(y_true, y_prob))
    if metric == "Precision":
        return float(precision_score(y_true, y_hat, zero_division=0))
    if metric == "Recall":
        return float(recall_score(y_true, y_hat, zero_division=0))
    if metric == "F1":
        return float(f1_score(y_true, y_hat, zero_division=0))

    raise ValueError(f"Unsupported metric: {metric}")


def fine_advantage(metric: str, fine_value: float, semantic_value: float) -> float:
    """
    Positive always means the fine-grained configuration is better.

    Higher-is-better metrics: fine - semantic
    Lower-is-better metrics (RMSE/MAE): semantic - fine
    """
    if metric in {"RMSE", "MAE"}:
        return semantic_value - fine_value
    return fine_value - semantic_value


# ============================================================
# 3. LOAD + STRICTLY PAIR OOF ROWS
# ============================================================

REQUIRED_OOF_COLUMNS = {
    "row_id",
    "student_id",
    "fold",
    "y_true",
    "y_pred",
    "model_name",
    "kc_name",
}


def read_oof(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing OOF file:\n  {path}\n"
            "Check OOF_ROOT and make sure the latest *_all.csv files have been generated."
        )

    df = pd.read_csv(path)

    missing = REQUIRED_OOF_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"{path} is missing required OOF columns: {sorted(missing)}.\n"
            "Use the latest OOF wrappers that save student_id and the standard seven columns."
        )

    if df["row_id"].duplicated().any():
        n_dup = int(df["row_id"].duplicated().sum())
        raise ValueError(f"{path} contains {n_dup} duplicate row_id values.")

    if df[["row_id", "student_id", "fold", "y_true", "y_pred"]].isna().any().any():
        raise ValueError(f"{path} contains missing values in required analysis columns.")

    # Every student should belong to exactly one fixed outer fold.
    per_student_fold_count = df.groupby("student_id")["fold"].nunique()
    if (per_student_fold_count > 1).any():
        bad = per_student_fold_count[per_student_fold_count > 1].index.tolist()[:10]
        raise ValueError(
            f"{path}: some students appear in multiple outer folds. Examples: {bad}"
        )

    return df


def load_and_pair(file_fine: Path, file_semantic: Path):
    fine = read_oof(file_fine)
    semantic = read_oof(file_semantic)

    n_fine = len(fine)
    n_semantic = len(semantic)

    fine_keep = fine[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    sem_keep = semantic[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    # --------------------------------------------------------
    # STRICT ROW-COVERAGE CHECK
    # --------------------------------------------------------
    # The two configurations must contain exactly the same
    # held-out interaction rows before any significance test.
    fine_row_ids = set(fine_keep["row_id"])
    semantic_row_ids = set(sem_keep["row_id"])

    missing_from_semantic = fine_row_ids - semantic_row_ids
    missing_from_fine = semantic_row_ids - fine_row_ids

    if missing_from_semantic or missing_from_fine:
        raise ValueError(
            "OOF row coverage does not match exactly between the two configurations.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Rows missing from Semantic: {len(missing_from_semantic)}\n"
            f"Rows missing from Fine: {len(missing_from_fine)}\n"
            "The paired significance test requires identical held-out "
            "interaction rows."
        )

    # --------------------------------------------------------
    # PAIR EXACTLY ON row_id
    # --------------------------------------------------------
    paired = fine_keep.merge(
        sem_keep,
        on="row_id",
        how="inner",
        suffixes=("_fine", "_semantic"),
        validate="one_to_one",
    ).sort_values("row_id").reset_index(drop=True)

    # This should now always hold because of the strict check above,
    # but keep it as an additional safeguard.
    if len(paired) != n_fine or len(paired) != n_semantic:
        raise ValueError(
            "Unexpected row-count mismatch after pairing.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Paired rows: {len(paired)}"
        )

    # --------------------------------------------------------
    # VERIFY TRUE LABELS MATCH
    # --------------------------------------------------------
    if not np.array_equal(
        paired["y_true_fine"].astype(int).to_numpy(),
        paired["y_true_semantic"].astype(int).to_numpy(),
    ):
        raise ValueError(
            "y_true mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # VERIFY STUDENT IDS MATCH
    # --------------------------------------------------------
    fine_students = paired["student_id_fine"].astype(str).to_numpy()
    semantic_students = paired["student_id_semantic"].astype(str).to_numpy()

    if not np.array_equal(fine_students, semantic_students):
        raise ValueError(
            "student_id mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # VERIFY OUTER FOLDS MATCH
    # --------------------------------------------------------
    fine_folds = paired["fold_fine"].astype(int).to_numpy()
    semantic_folds = paired["fold_semantic"].astype(int).to_numpy()

    if not np.array_equal(fine_folds, semantic_folds):
        raise ValueError(
            "Outer-fold mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # FINAL ARRAYS
    # --------------------------------------------------------
    y_true = paired["y_true_fine"].astype(int).to_numpy()
    pred_fine = paired["y_pred_fine"].astype(float).to_numpy()
    pred_semantic = paired["y_pred_semantic"].astype(float).to_numpy()
    students = fine_students
    folds = fine_folds

    diagnostics = {
        "Rows_Fine": n_fine,
        "Rows_Semantic": n_semantic,
        "Rows_Common": len(paired),
        "Students_Common": int(pd.Series(students).nunique()),
        "Fine_Coverage_in_Common": len(paired) / n_fine,
        "Semantic_Coverage_in_Common": len(paired) / n_semantic,
    }

    return (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    )


# ============================================================
# 4. STUDENT-CLUSTER BOOTSTRAP CI
# ============================================================

def build_fold_student_index(students, folds):
    """Map each outer fold -> student -> row indices."""
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    result = {}
    for fold in sorted(np.unique(folds)):
        fold_mask = folds == fold
        fold_students = np.unique(students[fold_mask])
        result[int(fold)] = {
            student: np.flatnonzero(fold_mask & (students == student))
            for student in fold_students
        }
    return result


def paired_student_cluster_bootstrap(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
):
    """
    Paired cluster bootstrap, stratified by outer fold.

    Within each outer fold, sample students with replacement and keep every row
    for each sampled student. Repeatedly sampled students are repeated in the
    bootstrap sample, as required by cluster bootstrap.
    """
    rng = np.random.default_rng(seed)
    fold_student_index = build_fold_student_index(students, folds)
    advantages = []

    for _ in range(n_bootstrap):
        sampled_index_parts = []

        for fold, student_map in fold_student_index.items():
            fold_students = np.array(list(student_map.keys()), dtype=object)
            sampled_students = rng.choice(
                fold_students,
                size=len(fold_students),
                replace=True,
            )

            sampled_index_parts.extend(
                student_map[student]
                for student in sampled_students
            )

        idx = np.concatenate(sampled_index_parts)

        fine_val = metric_value(metric, y_true[idx], pred_fine[idx])
        sem_val = metric_value(metric, y_true[idx], pred_semantic[idx])

        if np.isnan(fine_val) or np.isnan(sem_val):
            continue

        advantages.append(fine_advantage(metric, fine_val, sem_val))

    advantages = np.asarray(advantages, dtype=float)
    if len(advantages) == 0:
        return {
            "bootstrap_n": 0,
            "bootstrap_mean_advantage_fine": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    return {
        "bootstrap_n": int(len(advantages)),
        "bootstrap_mean_advantage_fine": float(np.mean(advantages)),
        "ci_low": float(np.percentile(advantages, 2.5)),
        "ci_high": float(np.percentile(advantages, 97.5)),
    }


# ============================================================
# 5. PAIRED STUDENT-CLUSTER PERMUTATION TEST
# ============================================================

def paired_student_cluster_permutation_test(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    n_permutations=N_PERMUTATIONS,
    seed=RANDOM_SEED,
):
    """
    Two-sided paired permutation test at the STUDENT level.

    Under the null that the two configurations are exchangeable, each student's
    complete vector of Fine/Semantic predictions is swapped as one block with
    probability 0.5. This keeps every student's repeated interactions together.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)

    fine_obs = metric_value(metric, y_true, pred_fine)
    semantic_obs = metric_value(metric, y_true, pred_semantic)
    observed_advantage = fine_advantage(metric, fine_obs, semantic_obs)

    unique_students, student_codes = np.unique(students, return_inverse=True)
    rng = np.random.default_rng(seed)

    extreme = 0
    valid = 0

    for _ in range(n_permutations):
        swap_by_student = rng.integers(0, 2, size=len(unique_students), dtype=np.int8)
        swap_rows = swap_by_student[student_codes].astype(bool)

        perm_fine = np.where(swap_rows, pred_semantic, pred_fine)
        perm_semantic = np.where(swap_rows, pred_fine, pred_semantic)

        fine_perm_val = metric_value(metric, y_true, perm_fine)
        sem_perm_val = metric_value(metric, y_true, perm_semantic)

        if np.isnan(fine_perm_val) or np.isnan(sem_perm_val):
            continue

        perm_advantage = fine_advantage(metric, fine_perm_val, sem_perm_val)
        valid += 1

        if abs(perm_advantage) >= abs(observed_advantage) - 1e-15:
            extreme += 1

    # Phipson-Smyth +1 correction avoids zero Monte Carlo p-values.
    p_value = (extreme + 1) / (valid + 1) if valid > 0 else np.nan

    return {
        "fine_value": fine_obs,
        "semantic_value": semantic_obs,
        "raw_diff_fine_minus_semantic": fine_obs - semantic_obs,
        "advantage_fine": observed_advantage,
        "permutation_n": valid,
        "p_value": float(p_value) if not np.isnan(p_value) else np.nan,
    }


# ============================================================
# 6. RUN ALL SELECTED FIRST-ATTEMPT COMPARISONS
# ============================================================

summary_rows = []

print("=" * 100)
print("FIRST-ATTEMPT SIGNIFICANCE: TOP FINE-GRAINED VS TOP SEMANTIC")
print("=" * 100)
print(f"OOF root: {OOF_ROOT.resolve()}")
print(f"Threshold: p > {THRESHOLD}")
print(f"Student-cluster bootstrap replicates: {N_BOOTSTRAP}")
print(f"Student-cluster permutation replicates: {N_PERMUTATIONS}")

for i, cfg in enumerate(COMPARISONS):
    metric = cfg["metric"]
    comparison_id = cfg["comparison_id"]

    print("\n" + "=" * 100)
    print(f"{comparison_id} | Metric: {metric}")
    print(f"Fine-grained: {cfg['fine_name']} [{cfg['fine_table_value']}]")
    print(f"Semantic:     {cfg['semantic_name']} [{cfg['semantic_table_value']}]")
    if cfg.get("note"):
        print(f"Note:         {cfg['note']}")
    print(f"Fine file:    {cfg['fine_file']}")
    print(f"Semantic file:{cfg['semantic_file']}")

    (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    ) = load_and_pair(cfg["fine_file"], cfg["semantic_file"])

    print(
        f"Paired rows: {diagnostics['Rows_Common']} | "
        f"Paired students: {diagnostics['Students_Common']} | "
        f"Fine coverage: {diagnostics['Fine_Coverage_in_Common']:.3%} | "
        f"Semantic coverage: {diagnostics['Semantic_Coverage_in_Common']:.3%}"
    )

    # Different deterministic seeds per comparison while preserving reproducibility.
    comparison_seed = RANDOM_SEED + i * 1000

    perm = paired_student_cluster_permutation_test(
        metric,
        y_true,
        pred_fine,
        pred_semantic,
        students,
        n_permutations=N_PERMUTATIONS,
        seed=comparison_seed,
    )

    boot = paired_student_cluster_bootstrap(
        metric,
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        n_bootstrap=N_BOOTSTRAP,
        seed=comparison_seed + 1,
    )

    print(f"Matched-OOF Fine value:      {perm['fine_value']:.6f}")
    print(f"Matched-OOF Semantic value:  {perm['semantic_value']:.6f}")
    print(f"Fine - Semantic raw diff:    {perm['raw_diff_fine_minus_semantic']:.6f}")
    print(
        "Fine advantage (positive=fine better): "
        f"{perm['advantage_fine']:.6f}"
    )
    print(
        "95% student-cluster bootstrap CI for Fine advantage: "
        f"[{boot['ci_low']:.6f}, {boot['ci_high']:.6f}]"
    )
    print(f"Student-cluster permutation p: {perm['p_value']:.6g}")

    summary_rows.append({
        "Comparison_ID": comparison_id,
        "Metric": metric,
        "Fine_Model": cfg["fine_name"],
        "Fine_Table_MeanSD": cfg["fine_table_value"],
        "Semantic_Model": cfg["semantic_name"],
        "Semantic_Table_MeanSD": cfg["semantic_table_value"],
        "Fine_File": str(cfg["fine_file"]),
        "Semantic_File": str(cfg["semantic_file"]),
        "Rows_Fine": diagnostics["Rows_Fine"],
        "Rows_Semantic": diagnostics["Rows_Semantic"],
        "Rows_Common": diagnostics["Rows_Common"],
        "Students_Common": diagnostics["Students_Common"],
        "Fine_Coverage_in_Common": diagnostics["Fine_Coverage_in_Common"],
        "Semantic_Coverage_in_Common": diagnostics["Semantic_Coverage_in_Common"],
        "Matched_OOF_Fine_Value": perm["fine_value"],
        "Matched_OOF_Semantic_Value": perm["semantic_value"],
        "Raw_Diff_Fine_minus_Semantic": perm["raw_diff_fine_minus_semantic"],
        "Advantage_Fine_PositiveMeansFineBetter": perm["advantage_fine"],
        "Bootstrap_Mean_Advantage_Fine": boot["bootstrap_mean_advantage_fine"],
        "Bootstrap_CI95_Low": boot["ci_low"],
        "Bootstrap_CI95_High": boot["ci_high"],
        "Permutation_P_Value": perm["p_value"],
        "Permutation_N": perm["permutation_n"],
        "Bootstrap_N": boot["bootstrap_n"],
        "Note": cfg.get("note", ""),
    })


# ============================================================
# 7. MULTIPLE-COMPARISON CORRECTION + SAVE
# ============================================================

summary_df = pd.DataFrame(summary_rows)

valid_p_mask = summary_df["Permutation_P_Value"].notna()
summary_df["Holm_P_Value"] = np.nan
summary_df["Significant_Holm_0.05"] = False

if valid_p_mask.any():
    rejected, p_holm, _, _ = multipletests(
        summary_df.loc[valid_p_mask, "Permutation_P_Value"].to_numpy(),
        alpha=ALPHA,
        method="holm",
    )
    summary_df.loc[valid_p_mask, "Holm_P_Value"] = p_holm
    summary_df.loc[valid_p_mask, "Significant_Holm_0.05"] = rejected

# CI-based indication is useful alongside the permutation p-value.
summary_df["CI_Excludes_Zero"] = (
    (summary_df["Bootstrap_CI95_Low"] > 0)
    | (summary_df["Bootstrap_CI95_High"] < 0)
)

output_file = "first_attempt_fine_vs_semantic_significance.csv"
summary_df.to_csv(output_file, index=False)

print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("=" * 100)

cols_to_print = [
    "Comparison_ID",
    "Metric",
    "Fine_Model",
    "Semantic_Model",
    "Rows_Common",
    "Students_Common",
    "Matched_OOF_Fine_Value",
    "Matched_OOF_Semantic_Value",
    "Advantage_Fine_PositiveMeansFineBetter",
    "Bootstrap_CI95_Low",
    "Bootstrap_CI95_High",
    "Permutation_P_Value",
    "Holm_P_Value",
    "Significant_Holm_0.05",
]

with pd.option_context(
    "display.max_columns", None,
    "display.width", 240,
    "display.max_colwidth", 55,
):
    print(summary_df[cols_to_print].to_string(index=False))

print(f"\nSaved summary to: {output_file}")

In [ ]:
# After Feedback Pooled Statistical Significance


"""
After Feedback statistical significance analysis:
best fine-grained vs best semantic DLKT configuration for each metric.

Selection basis
---------------
Configurations are selected from the after-feedback results table reported in
the corresponding results table.

Fine-grained KC family:
    actionableelementid, itemid, itemsetid, exerciseid

Semantic KC family:
    propertyexercisetype, linguisticconstructs, propertyid

Exercise ID is treated as fine-grained, consistently with the first-attempt
analysis.

Inference
---------
OOF rows are repeated observations nested within students. Therefore inference
is student-cluster aware:

  1) paired STUDENT-CLUSTER permutation tests for two-sided p-values;
  2) paired STUDENT-CLUSTER bootstrap for 95% confidence intervals.

Students are kept intact during resampling/permutation. Comparisons are paired
on common row_id values only, and student_id, y_true, and fold must agree
between the two OOF files.

Holm correction is applied across the seven metric-wise comparisons.

Expected OOF layout
-------------------
    AKT_AF/oof_AKT_<kc_name>_all.csv
    SAKT_AF/oof_SAKT_<kc_name>_all.csv
    DKVMN_AF/oof_DKVMN_<kc_name>_all.csv
    SKVMN_AF/oof_SKVMN_<kc_name>_all.csv
    DKT_AF/oof_DKT_<kc_name>_all.csv
    DKTForget_AF/oof_DKTForget_<kc_name>_all.csv

Required OOF columns:
    row_id, student_id, fold, y_true, y_pred, model_name, kc_name

RESOLVED MAE TIE
----------------
The rounded table showed a tie for the best fine-grained MAE, but the
unrounded fold-average values reported in the corresponding results table resolve it:
    DKT + Item Set ID = 0.195803 ± 0.002680
    AKT + Exercise ID = 0.195633 ± 0.007185

Lower MAE is better, so AKT + Exercise ID is retained as the single
fine-grained MAE winner. The final analysis contains exactly seven tests.
"""

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
)
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Run this script from the directory containing AKT_AF, DKTForget_AF, etc.,
# or change OOF_ROOT to the parent directory containing those folders.
OOF_ROOT = Path(".")

# Match the KT wrappers: probability must be STRICTLY greater than 0.5.
THRESHOLD = 0.5

# Student-cluster resampling settings.
# For a quick smoke test, temporarily reduce these to e.g. 200.
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 10000
RANDOM_SEED = 42
ALPHA = 0.05

# Exercise ID is included in the fine-grained family.
FINE_GRAINED_KCS = {
    "actionableelementid",
    "itemid",
    "itemsetid",
    "exerciseid",
}

SEMANTIC_KCS = {
    "propertyexercisetype",
    "linguisticconstructs",
    "propertyid",
}


def oof_path(folder: str, model: str, kc: str) -> Path:
    """Build the OOF path, using student-ID-fixed files for AKT."""
    if model == "AKT":
        return OOF_ROOT / folder / f"oof_{model}_{kc}_all_with_student_id.csv"

    return OOF_ROOT / folder / f"oof_{model}_{kc}_all.csv"


# Exactly one fine-grained-vs-semantic comparison per metric (7 total).
COMPARISONS = [
    {
        "comparison_id": "AUC",
        "metric": "AUC",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.871 ± 0.012",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.851 ± 0.009",
    },
    {
        "comparison_id": "Accuracy",
        "metric": "Accuracy",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.856 ± 0.006",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.833 ± 0.006",
    },
    {
        "comparison_id": "RMSE",
        "metric": "RMSE",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.336 ± 0.009",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.353 ± 0.007",
    },
    {
        "comparison_id": "MAE",
        "metric": "MAE",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.196 ± 0.007",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_AF", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.216 ± 0.007",
    },
    {
        "comparison_id": "Precision",
        "metric": "Precision",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.880 ± 0.008",
        "semantic_name": "AKT + Linguistic Constructs",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "linguisticconstructs"
        ),
        "semantic_table_value": "0.857 ± 0.003",
    },
    {
        "comparison_id": "Recall",
        "metric": "Recall",
        "fine_name": "DKVMN + Exercise ID",
        "fine_file": oof_path(
            "DKVMN_AF", "DKVMN", "exerciseid"
        ),
        "fine_table_value": "0.940 ± 0.007",
        "semantic_name": "SKVMN + Property ID",
        "semantic_file": oof_path(
            "SKVMN_AF", "SKVMN", "propertyid"
        ),
        "semantic_table_value": "0.935 ± 0.009",
    },
    {
        "comparison_id": "F1",
        "metric": "F1",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.905 ± 0.003",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.892 ± 0.004",
    },
]

if len(COMPARISONS) != 7:
    raise RuntimeError(
        f"Expected exactly 7 comparisons, got {len(COMPARISONS)}."
    )


# ============================================================
# 2. METRICS
# ============================================================

def rmse_score(y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_prob) ** 2)))


def metric_value(metric: str, y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    if metric == "AUC":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))

    # Match the model wrappers exactly: p > 0.5 -> 1, otherwise 0.
    y_hat = (y_prob > THRESHOLD).astype(int)

    if metric == "Accuracy":
        return float(accuracy_score(y_true, y_hat))
    if metric == "RMSE":
        return rmse_score(y_true, y_prob)
    if metric == "MAE":
        return float(mean_absolute_error(y_true, y_prob))
    if metric == "Precision":
        return float(precision_score(y_true, y_hat, zero_division=0))
    if metric == "Recall":
        return float(recall_score(y_true, y_hat, zero_division=0))
    if metric == "F1":
        return float(f1_score(y_true, y_hat, zero_division=0))

    raise ValueError(f"Unsupported metric: {metric}")


def fine_advantage(metric: str, fine_value: float, semantic_value: float) -> float:
    """
    Positive always means the fine-grained configuration is better.

    Higher-is-better metrics: fine - semantic
    Lower-is-better metrics (RMSE/MAE): semantic - fine
    """
    if metric in {"RMSE", "MAE"}:
        return semantic_value - fine_value
    return fine_value - semantic_value


# ============================================================
# 3. LOAD + STRICTLY PAIR OOF ROWS
# ============================================================

REQUIRED_OOF_COLUMNS = {
    "row_id",
    "student_id",
    "fold",
    "y_true",
    "y_pred",
    "model_name",
    "kc_name",
}


def read_oof(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing OOF file:\n  {path}\n"
            "Check OOF_ROOT and make sure the latest *_all.csv files have been generated."
        )

    df = pd.read_csv(path)

    missing = REQUIRED_OOF_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"{path} is missing required OOF columns: {sorted(missing)}.\n"
            "Use the latest OOF wrappers that save student_id and the standard seven columns."
        )

    if df["row_id"].duplicated().any():
        n_dup = int(df["row_id"].duplicated().sum())
        raise ValueError(f"{path} contains {n_dup} duplicate row_id values.")

    if df[["row_id", "student_id", "fold", "y_true", "y_pred"]].isna().any().any():
        raise ValueError(f"{path} contains missing values in required analysis columns.")

    # Every student should belong to exactly one fixed outer fold.
    per_student_fold_count = df.groupby("student_id")["fold"].nunique()
    if (per_student_fold_count > 1).any():
        bad = per_student_fold_count[per_student_fold_count > 1].index.tolist()[:10]
        raise ValueError(
            f"{path}: some students appear in multiple outer folds. Examples: {bad}"
        )

    return df


def load_and_pair(file_fine: Path, file_semantic: Path):
    fine = read_oof(file_fine)
    semantic = read_oof(file_semantic)

    n_fine = len(fine)
    n_semantic = len(semantic)

    fine_keep = fine[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    sem_keep = semantic[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    # --------------------------------------------------------
    # STRICT ROW-COVERAGE CHECK
    # --------------------------------------------------------
    # Both configurations must contain exactly the same
    # held-out interaction rows.
    fine_row_ids = set(fine_keep["row_id"])
    semantic_row_ids = set(sem_keep["row_id"])

    missing_from_semantic = fine_row_ids - semantic_row_ids
    missing_from_fine = semantic_row_ids - fine_row_ids

    if missing_from_semantic or missing_from_fine:
        raise ValueError(
            "OOF row coverage does not match exactly between the two configurations.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Rows missing from Semantic: {len(missing_from_semantic)}\n"
            f"Rows missing from Fine: {len(missing_from_fine)}\n"
            "The paired significance test requires identical held-out "
            "interaction rows."
        )

    # --------------------------------------------------------
    # PAIR EXACTLY ON row_id
    # --------------------------------------------------------
    paired = fine_keep.merge(
        sem_keep,
        on="row_id",
        how="inner",
        suffixes=("_fine", "_semantic"),
        validate="one_to_one",
    ).sort_values("row_id").reset_index(drop=True)

    # Additional safeguard.
    if len(paired) != n_fine or len(paired) != n_semantic:
        raise ValueError(
            "Unexpected row-count mismatch after pairing.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Paired rows: {len(paired)}"
        )

    # --------------------------------------------------------
    # VERIFY TRUE LABELS MATCH
    # --------------------------------------------------------
    if not np.array_equal(
        paired["y_true_fine"].astype(int).to_numpy(),
        paired["y_true_semantic"].astype(int).to_numpy(),
    ):
        raise ValueError(
            "y_true mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # VERIFY STUDENT IDS MATCH
    # --------------------------------------------------------
    fine_students = paired["student_id_fine"].astype(str).to_numpy()
    semantic_students = paired["student_id_semantic"].astype(str).to_numpy()

    if not np.array_equal(fine_students, semantic_students):
        raise ValueError(
            "student_id mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # VERIFY OUTER FOLDS MATCH
    # --------------------------------------------------------
    fine_folds = paired["fold_fine"].astype(int).to_numpy()
    semantic_folds = paired["fold_semantic"].astype(int).to_numpy()

    if not np.array_equal(fine_folds, semantic_folds):
        raise ValueError(
            "Outer-fold mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    # --------------------------------------------------------
    # FINAL ARRAYS
    # --------------------------------------------------------
    y_true = paired["y_true_fine"].astype(int).to_numpy()
    pred_fine = paired["y_pred_fine"].astype(float).to_numpy()
    pred_semantic = paired["y_pred_semantic"].astype(float).to_numpy()
    students = fine_students
    folds = fine_folds

    diagnostics = {
        "Rows_Fine": n_fine,
        "Rows_Semantic": n_semantic,
        "Rows_Common": len(paired),
        "Students_Common": int(pd.Series(students).nunique()),
        "Fine_Coverage_in_Common": len(paired) / n_fine,
        "Semantic_Coverage_in_Common": len(paired) / n_semantic,
    }

    return (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    )

# ============================================================
# 4. STUDENT-CLUSTER BOOTSTRAP CI
# ============================================================

def build_fold_student_index(students, folds):
    """Map each outer fold -> student -> row indices."""
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    result = {}
    for fold in sorted(np.unique(folds)):
        fold_mask = folds == fold
        fold_students = np.unique(students[fold_mask])
        result[int(fold)] = {
            student: np.flatnonzero(fold_mask & (students == student))
            for student in fold_students
        }
    return result


def paired_student_cluster_bootstrap(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
):
    """
    Paired cluster bootstrap, stratified by outer fold.

    Within each outer fold, sample students with replacement and keep every row
    for each sampled student. Repeatedly sampled students are repeated in the
    bootstrap sample, as required by cluster bootstrap.
    """
    rng = np.random.default_rng(seed)
    fold_student_index = build_fold_student_index(students, folds)
    advantages = []

    for _ in range(n_bootstrap):
        sampled_index_parts = []

        for fold, student_map in fold_student_index.items():
            fold_students = np.array(list(student_map.keys()), dtype=object)
            sampled_students = rng.choice(
                fold_students,
                size=len(fold_students),
                replace=True,
            )

            sampled_index_parts.extend(
                student_map[student]
                for student in sampled_students
            )

        idx = np.concatenate(sampled_index_parts)

        fine_val = metric_value(metric, y_true[idx], pred_fine[idx])
        sem_val = metric_value(metric, y_true[idx], pred_semantic[idx])

        if np.isnan(fine_val) or np.isnan(sem_val):
            continue

        advantages.append(fine_advantage(metric, fine_val, sem_val))

    advantages = np.asarray(advantages, dtype=float)
    if len(advantages) == 0:
        return {
            "bootstrap_n": 0,
            "bootstrap_mean_advantage_fine": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    return {
        "bootstrap_n": int(len(advantages)),
        "bootstrap_mean_advantage_fine": float(np.mean(advantages)),
        "ci_low": float(np.percentile(advantages, 2.5)),
        "ci_high": float(np.percentile(advantages, 97.5)),
    }


# ============================================================
# 5. PAIRED STUDENT-CLUSTER PERMUTATION TEST
# ============================================================

def paired_student_cluster_permutation_test(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    n_permutations=N_PERMUTATIONS,
    seed=RANDOM_SEED,
):
    """
    Two-sided paired permutation test at the STUDENT level.

    Under the null that the two configurations are exchangeable, each student's
    complete vector of Fine/Semantic predictions is swapped as one block with
    probability 0.5. This keeps every student's repeated interactions together.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)

    fine_obs = metric_value(metric, y_true, pred_fine)
    semantic_obs = metric_value(metric, y_true, pred_semantic)
    observed_advantage = fine_advantage(metric, fine_obs, semantic_obs)

    unique_students, student_codes = np.unique(students, return_inverse=True)
    rng = np.random.default_rng(seed)

    extreme = 0
    valid = 0

    for _ in range(n_permutations):
        swap_by_student = rng.integers(0, 2, size=len(unique_students), dtype=np.int8)
        swap_rows = swap_by_student[student_codes].astype(bool)

        perm_fine = np.where(swap_rows, pred_semantic, pred_fine)
        perm_semantic = np.where(swap_rows, pred_fine, pred_semantic)

        fine_perm_val = metric_value(metric, y_true, perm_fine)
        sem_perm_val = metric_value(metric, y_true, perm_semantic)

        if np.isnan(fine_perm_val) or np.isnan(sem_perm_val):
            continue

        perm_advantage = fine_advantage(metric, fine_perm_val, sem_perm_val)
        valid += 1

        if abs(perm_advantage) >= abs(observed_advantage) - 1e-15:
            extreme += 1

    # Phipson-Smyth +1 correction avoids zero Monte Carlo p-values.
    p_value = (extreme + 1) / (valid + 1) if valid > 0 else np.nan

    return {
        "fine_value": fine_obs,
        "semantic_value": semantic_obs,
        "raw_diff_fine_minus_semantic": fine_obs - semantic_obs,
        "advantage_fine": observed_advantage,
        "permutation_n": valid,
        "p_value": float(p_value) if not np.isnan(p_value) else np.nan,
    }


# ============================================================
# 6. RUN ALL SELECTED AFTER-FEEDBACK COMPARISONS
# ============================================================

summary_rows = []

print("=" * 100)
print("AFTER-FEEDBACK SIGNIFICANCE: TOP FINE-GRAINED VS TOP SEMANTIC")
print("=" * 100)
print(f"OOF root: {OOF_ROOT.resolve()}")
print(f"Threshold: p > {THRESHOLD}")
print(f"Student-cluster bootstrap replicates: {N_BOOTSTRAP}")
print(f"Student-cluster permutation replicates: {N_PERMUTATIONS}")

for i, cfg in enumerate(COMPARISONS):
    metric = cfg["metric"]
    comparison_id = cfg["comparison_id"]

    print("\n" + "=" * 100)
    print(f"{comparison_id} | Metric: {metric}")
    print(f"Fine-grained: {cfg['fine_name']} [{cfg['fine_table_value']}]")
    print(f"Semantic:     {cfg['semantic_name']} [{cfg['semantic_table_value']}]")
    if cfg.get("note"):
        print(f"Note:         {cfg['note']}")
    print(f"Fine file:    {cfg['fine_file']}")
    print(f"Semantic file:{cfg['semantic_file']}")

    (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    ) = load_and_pair(cfg["fine_file"], cfg["semantic_file"])

    print(
        f"Paired rows: {diagnostics['Rows_Common']} | "
        f"Paired students: {diagnostics['Students_Common']} | "
        f"Fine coverage: {diagnostics['Fine_Coverage_in_Common']:.3%} | "
        f"Semantic coverage: {diagnostics['Semantic_Coverage_in_Common']:.3%}"
    )

    # Different deterministic seeds per comparison while preserving reproducibility.
    comparison_seed = RANDOM_SEED + i * 1000

    perm = paired_student_cluster_permutation_test(
        metric,
        y_true,
        pred_fine,
        pred_semantic,
        students,
        n_permutations=N_PERMUTATIONS,
        seed=comparison_seed,
    )

    boot = paired_student_cluster_bootstrap(
        metric,
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        n_bootstrap=N_BOOTSTRAP,
        seed=comparison_seed + 1,
    )

    print(f"Matched-OOF Fine value:      {perm['fine_value']:.6f}")
    print(f"Matched-OOF Semantic value:  {perm['semantic_value']:.6f}")
    print(f"Fine - Semantic raw diff:    {perm['raw_diff_fine_minus_semantic']:.6f}")
    print(
        "Fine advantage (positive=fine better): "
        f"{perm['advantage_fine']:.6f}"
    )
    print(
        "95% student-cluster bootstrap CI for Fine advantage: "
        f"[{boot['ci_low']:.6f}, {boot['ci_high']:.6f}]"
    )
    print(f"Student-cluster permutation p: {perm['p_value']:.6g}")

    summary_rows.append({
        "Comparison_ID": comparison_id,
        "Metric": metric,
        "Fine_Model": cfg["fine_name"],
        "Fine_Table_MeanSD": cfg["fine_table_value"],
        "Semantic_Model": cfg["semantic_name"],
        "Semantic_Table_MeanSD": cfg["semantic_table_value"],
        "Fine_File": str(cfg["fine_file"]),
        "Semantic_File": str(cfg["semantic_file"]),
        "Rows_Fine": diagnostics["Rows_Fine"],
        "Rows_Semantic": diagnostics["Rows_Semantic"],
        "Rows_Common": diagnostics["Rows_Common"],
        "Students_Common": diagnostics["Students_Common"],
        "Fine_Coverage_in_Common": diagnostics["Fine_Coverage_in_Common"],
        "Semantic_Coverage_in_Common": diagnostics["Semantic_Coverage_in_Common"],
        "Matched_OOF_Fine_Value": perm["fine_value"],
        "Matched_OOF_Semantic_Value": perm["semantic_value"],
        "Raw_Diff_Fine_minus_Semantic": perm["raw_diff_fine_minus_semantic"],
        "Advantage_Fine_PositiveMeansFineBetter": perm["advantage_fine"],
        "Bootstrap_Mean_Advantage_Fine": boot["bootstrap_mean_advantage_fine"],
        "Bootstrap_CI95_Low": boot["ci_low"],
        "Bootstrap_CI95_High": boot["ci_high"],
        "Permutation_P_Value": perm["p_value"],
        "Permutation_N": perm["permutation_n"],
        "Bootstrap_N": boot["bootstrap_n"],
        "Note": cfg.get("note", ""),
    })


# ============================================================
# 7. MULTIPLE-COMPARISON CORRECTION + SAVE
# ============================================================

summary_df = pd.DataFrame(summary_rows)

valid_p_mask = summary_df["Permutation_P_Value"].notna()
summary_df["Holm_P_Value"] = np.nan
summary_df["Significant_Holm_0.05"] = False

if valid_p_mask.any():
    rejected, p_holm, _, _ = multipletests(
        summary_df.loc[valid_p_mask, "Permutation_P_Value"].to_numpy(),
        alpha=ALPHA,
        method="holm",
    )
    summary_df.loc[valid_p_mask, "Holm_P_Value"] = p_holm
    summary_df.loc[valid_p_mask, "Significant_Holm_0.05"] = rejected

# CI-based indication is useful alongside the permutation p-value.
summary_df["CI_Excludes_Zero"] = (
    (summary_df["Bootstrap_CI95_Low"] > 0)
    | (summary_df["Bootstrap_CI95_High"] < 0)
)

output_file = "after_feedback_fine_vs_semantic_significance.csv"
summary_df.to_csv(output_file, index=False)

print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("=" * 100)

cols_to_print = [
    "Comparison_ID",
    "Metric",
    "Fine_Model",
    "Semantic_Model",
    "Rows_Common",
    "Students_Common",
    "Matched_OOF_Fine_Value",
    "Matched_OOF_Semantic_Value",
    "Advantage_Fine_PositiveMeansFineBetter",
    "Bootstrap_CI95_Low",
    "Bootstrap_CI95_High",
    "Permutation_P_Value",
    "Holm_P_Value",
    "Significant_Holm_0.05",
]

with pd.option_context(
    "display.max_columns", None,
    "display.width", 240,
    "display.max_colwidth", 55,
):
    print(summary_df[cols_to_print].to_string(index=False))

print(f"\nSaved summary to: {output_file}")

In [ ]:
# First Attempt Averaged Statistical Significance 

"""
First Attempt statistical significance analysis:
best fine-grained vs best semantic DLKT configuration for each metric.

STATISTIC USED
--------------
The reported performance statistic is the UNWEIGHTED MEAN of the metric
calculated separately in each outer test fold. This matches the "Average Score"
reported in the main results table.

Inference is still performed from paired out-of-fold predictions while respecting
student-level dependence:

  1) two-sided paired STUDENT-CLUSTER permutation test for p-values;
  2) fold-stratified paired STUDENT-CLUSTER bootstrap for 95% confidence intervals;
  3) Holm correction across the seven metric-wise tests within this prediction stage.

For every observed/permuted/bootstrap dataset, the statistic is computed in the
same way:

    metric in fold 1
    metric in fold 2
    metric in fold 3
        -> unweighted mean across outer folds

Students remain intact during permutation/bootstrap. Fine and semantic predictions
are paired on exactly the same row_id values, and student_id, y_true, and fold must
agree between the two OOF files.

Expected OOF layout
-------------------
    AKT_FA/oof_AKT_<kc_name>_all_with_student_id.csv
    SAKT_FA/oof_SAKT_<kc_name>_all.csv
    DKVMN_FA/oof_DKVMN_<kc_name>_all.csv
    SKVMN_FA/oof_SKVMN_<kc_name>_all.csv
    DKT_FA/oof_DKT_<kc_name>_all.csv
    DKTForget_FA/oof_DKTForget_<kc_name>_all.csv

Required OOF columns:
    row_id, student_id, fold, y_true, y_pred, model_name, kc_name
"""

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
)
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Run this script from the directory containing AKT_FA, DKTForget_FA, etc.,
# or change OOF_ROOT to the parent directory containing those folders.
OOF_ROOT = Path(".")

# Match the KT wrappers exactly: probability must be STRICTLY greater than 0.5.
THRESHOLD = 0.5

# Your outer CV uses three folds. This is checked in every paired comparison.
EXPECTED_N_OUTER_FOLDS = 3

# Student-cluster resampling settings.
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 10000
RANDOM_SEED = 42
ALPHA = 0.05

# Number of decimals used by the displayed main results table.
TABLE_DECIMALS = 3

# Exercise ID is treated as fine-grained.
FINE_GRAINED_KCS = {
    "actionableelementid",
    "itemid",
    "itemsetid",
    "exerciseid",
}

SEMANTIC_KCS = {
    "propertyexercisetype",
    "linguisticconstructs",
    "propertyid",
}


def oof_path(folder: str, model: str, kc: str) -> Path:
    """Build OOF path, using the student-ID-fixed AKT files."""
    if model == "AKT":
        return OOF_ROOT / folder / f"oof_{model}_{kc}_all_with_student_id.csv"
    return OOF_ROOT / folder / f"oof_{model}_{kc}_all.csv"


# Best fine-grained and semantic configuration for each metric,
# selected from the FIRST-ATTEMPT Average Score table.
# Values below are displayed to three decimals only.
COMPARISONS = [
    {
        "comparison_id": "AUC",
        "metric": "AUC",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.825 ± 0.003",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.821 ± 0.005",
    },
    {
        "comparison_id": "Accuracy",
        "metric": "Accuracy",
        "fine_name": "DKT+Forget + Item ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "itemid"
        ),
        "fine_table_value": "0.790 ± 0.004",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.783 ± 0.002",
    },
    {
        "comparison_id": "RMSE",
        "metric": "RMSE",
        "fine_name": "DKT+Forget + Item ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "itemid"
        ),
        "fine_table_value": "0.387 ± 0.004",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.389 ± 0.002",
    },
    {
        "comparison_id": "MAE",
        "metric": "MAE",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.255 ± 0.003",
        "semantic_name": "DKT+Forget + Linguistic Constructs",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "linguisticconstructs"
        ),
        "semantic_table_value": "0.263 ± 0.005",
    },
    {
        "comparison_id": "Precision",
        "metric": "Precision",
        "fine_name": "DKT+Forget + Actionable Element ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "actionableelementid"
        ),
        "fine_table_value": "0.831 ± 0.008",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.823 ± 0.008",
    },
    {
        "comparison_id": "Recall",
        "metric": "Recall",
        "fine_name": "AKT + Item Set ID",
        "fine_file": oof_path(
            "AKT_FA", "AKT", "itemsetid"
        ),
        "fine_table_value": "0.919 ± 0.008",
        "semantic_name": "DKVMN + Property ID",
        "semantic_file": oof_path(
            "DKVMN_FA", "DKVMN", "propertyid"
        ),
        "semantic_table_value": "0.919 ± 0.004",
    },
    {
        "comparison_id": "F1",
        "metric": "F1",
        "fine_name": "DKT+Forget + Item ID",
        "fine_file": oof_path(
            "DKTForget_FA", "DKTForget", "itemid"
        ),
        "fine_table_value": "0.859 ± 0.004",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_FA", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.855 ± 0.001",
    },
]

if len(COMPARISONS) != 7:
    raise RuntimeError(
        f"Expected exactly 7 comparisons, got {len(COMPARISONS)}."
    )


# ============================================================
# 2. METRICS + FOLD-AVERAGED STATISTIC
# ============================================================

def rmse_score(y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_prob) ** 2)))


def metric_value(metric: str, y_true, y_prob) -> float:
    """Calculate one metric on one set of interaction-level predictions."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(y_true) == 0:
        return np.nan

    if metric == "AUC":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))

    # Match model wrappers exactly: p > 0.5 -> 1, otherwise 0.
    y_hat = (y_prob > THRESHOLD).astype(int)

    if metric == "Accuracy":
        return float(accuracy_score(y_true, y_hat))
    if metric == "RMSE":
        return rmse_score(y_true, y_prob)
    if metric == "MAE":
        return float(mean_absolute_error(y_true, y_prob))
    if metric == "Precision":
        return float(precision_score(y_true, y_hat, zero_division=0))
    if metric == "Recall":
        return float(recall_score(y_true, y_hat, zero_division=0))
    if metric == "F1":
        return float(f1_score(y_true, y_hat, zero_division=0))

    raise ValueError(f"Unsupported metric: {metric}")


def fold_metric_values(metric: str, y_true, y_prob, folds) -> np.ndarray:
    """
    Calculate the requested metric separately in every outer fold.

    The returned values are the same type of fold-level scores used to build
    the Average Score (mean ± sample SD) in the main results table.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    folds = np.asarray(folds, dtype=int)

    if not (len(y_true) == len(y_prob) == len(folds)):
        raise ValueError("y_true, y_prob, and folds must have the same length.")

    unique_folds = np.sort(np.unique(folds))
    if len(unique_folds) == 0:
        return np.asarray([], dtype=float)

    values = []
    for fold in unique_folds:
        mask = folds == fold
        value = metric_value(metric, y_true[mask], y_prob[mask])
        if np.isnan(value):
            return np.asarray([], dtype=float)
        values.append(value)

    return np.asarray(values, dtype=float)


def fold_metric_summary(metric: str, y_true, y_prob, folds):
    """
    Return the unweighted mean and sample SD of the outer-fold metric values.

    Sample SD uses ddof=1, matching the SD convention used in the supplied
    first-attempt Average Score table.
    """
    values = fold_metric_values(metric, y_true, y_prob, folds)

    if len(values) == 0:
        return np.nan, np.nan, values

    mean_value = float(np.mean(values))
    sd_value = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
    return mean_value, sd_value, values


def fold_average_metric(metric: str, y_true, y_prob, folds) -> float:
    """Unweighted mean of the requested metric across outer folds."""
    mean_value, _, _ = fold_metric_summary(metric, y_true, y_prob, folds)
    return mean_value


def raw_difference(fine_value: float, semantic_value: float) -> float:
    """Raw Model A - Model B difference."""
    return float(fine_value - semantic_value)


def fine_advantage(metric: str, fine_value: float, semantic_value: float) -> float:
    """
    Direction-standardized difference.

    Positive always means the fine-grained configuration is better.
      - Higher-is-better metrics: fine - semantic
      - Lower-is-better metrics (RMSE/MAE): semantic - fine
    """
    if metric in {"RMSE", "MAE"}:
        return float(semantic_value - fine_value)
    return float(fine_value - semantic_value)


def parse_table_mean_sd(value: str):
    """Parse a string such as '0.825 ± 0.003'."""
    parts = value.split("±")
    if len(parts) != 2:
        raise ValueError(f"Could not parse table value: {value!r}")
    return float(parts[0].strip()), float(parts[1].strip())


def verify_against_table(
    label: str,
    computed_mean: float,
    computed_sd: float,
    table_value: str,
    decimals: int = TABLE_DECIMALS,
):
    """
    Verify that OOF-derived fold mean and sample SD reproduce the displayed
    Average Score after rounding to the table precision.

    This is a safeguard only; the hard-coded table strings are never used as
    inputs to the significance calculations.
    """
    table_mean, table_sd = parse_table_mean_sd(table_value)

    mean_matches = f"{computed_mean:.{decimals}f}" == f"{table_mean:.{decimals}f}"
    sd_matches = f"{computed_sd:.{decimals}f}" == f"{table_sd:.{decimals}f}"

    if not (mean_matches and sd_matches):
        raise ValueError(
            f"{label}: OOF-derived fold summary does not match the supplied "
            f"Average Score after rounding to {decimals} decimals.\n"
            f"Computed: {computed_mean:.6f} ± {computed_sd:.6f}\n"
            f"Table:    {table_value}\n"
            "Check that the correct OOF files/folds are being used."
        )


# ============================================================
# 3. LOAD + STRICTLY PAIR OOF ROWS
# ============================================================

REQUIRED_OOF_COLUMNS = {
    "row_id",
    "student_id",
    "fold",
    "y_true",
    "y_pred",
    "model_name",
    "kc_name",
}


def read_oof(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing OOF file:\n  {path}\n"
            "Check OOF_ROOT and make sure the expected *_all.csv file exists."
        )

    df = pd.read_csv(path)

    missing = REQUIRED_OOF_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"{path} is missing required OOF columns: {sorted(missing)}.\n"
            "Required columns are row_id, student_id, fold, y_true, y_pred, "
            "model_name, and kc_name."
        )

    if df["row_id"].duplicated().any():
        n_dup = int(df["row_id"].duplicated().sum())
        raise ValueError(f"{path} contains {n_dup} duplicate row_id values.")

    analysis_cols = ["row_id", "student_id", "fold", "y_true", "y_pred"]
    if df[analysis_cols].isna().any().any():
        raise ValueError(
            f"{path} contains missing values in required analysis columns."
        )

    # Every student must belong to exactly one fixed outer fold.
    per_student_fold_count = df.groupby("student_id")["fold"].nunique()
    if (per_student_fold_count > 1).any():
        bad = per_student_fold_count[per_student_fold_count > 1].index.tolist()[:10]
        raise ValueError(
            f"{path}: some students appear in multiple outer folds. Examples: {bad}"
        )

    n_folds = int(df["fold"].nunique())
    if n_folds != EXPECTED_N_OUTER_FOLDS:
        raise ValueError(
            f"{path}: expected {EXPECTED_N_OUTER_FOLDS} outer folds, "
            f"but found {n_folds}: {sorted(df['fold'].unique().tolist())}"
        )

    return df


def load_and_pair(file_fine: Path, file_semantic: Path):
    fine = read_oof(file_fine)
    semantic = read_oof(file_semantic)

    n_fine = len(fine)
    n_semantic = len(semantic)

    fine_keep = fine[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    sem_keep = semantic[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    # --------------------------------------------------------
    # STRICT ROW-COVERAGE CHECK
    # --------------------------------------------------------
    # Both configurations must contain exactly the same held-out interactions.
    fine_row_ids = set(fine_keep["row_id"])
    semantic_row_ids = set(sem_keep["row_id"])

    missing_from_semantic = fine_row_ids - semantic_row_ids
    missing_from_fine = semantic_row_ids - fine_row_ids

    if missing_from_semantic or missing_from_fine:
        raise ValueError(
            "OOF row coverage does not match exactly between the two configurations.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Rows missing from Semantic: {len(missing_from_semantic)}\n"
            f"Rows missing from Fine: {len(missing_from_fine)}\n"
            "The paired significance test requires identical held-out "
            "interaction rows."
        )

    # --------------------------------------------------------
    # PAIR EXACTLY ON row_id
    # --------------------------------------------------------
    paired = fine_keep.merge(
        sem_keep,
        on="row_id",
        how="inner",
        suffixes=("_fine", "_semantic"),
        validate="one_to_one",
    ).sort_values("row_id").reset_index(drop=True)

    if len(paired) != n_fine or len(paired) != n_semantic:
        raise ValueError(
            "Unexpected row-count mismatch after pairing.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Paired rows: {len(paired)}"
        )

    # --------------------------------------------------------
    # VERIFY y_true, student_id, AND fold MATCH
    # --------------------------------------------------------
    fine_y = paired["y_true_fine"].astype(int).to_numpy()
    semantic_y = paired["y_true_semantic"].astype(int).to_numpy()
    if not np.array_equal(fine_y, semantic_y):
        raise ValueError(
            "y_true mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    fine_students = paired["student_id_fine"].astype(str).to_numpy()
    semantic_students = paired["student_id_semantic"].astype(str).to_numpy()
    if not np.array_equal(fine_students, semantic_students):
        raise ValueError(
            "student_id mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    fine_folds = paired["fold_fine"].astype(int).to_numpy()
    semantic_folds = paired["fold_semantic"].astype(int).to_numpy()
    if not np.array_equal(fine_folds, semantic_folds):
        raise ValueError(
            "Outer-fold mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    y_true = fine_y
    pred_fine = paired["y_pred_fine"].astype(float).to_numpy()
    pred_semantic = paired["y_pred_semantic"].astype(float).to_numpy()
    students = fine_students
    folds = fine_folds

    diagnostics = {
        "Rows_Fine": n_fine,
        "Rows_Semantic": n_semantic,
        "Rows_Common": len(paired),
        "Students_Common": int(pd.Series(students).nunique()),
        "Outer_Folds": int(pd.Series(folds).nunique()),
        "Fine_Coverage_in_Common": len(paired) / n_fine,
        "Semantic_Coverage_in_Common": len(paired) / n_semantic,
    }

    return (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    )


# ============================================================
# 4. FOLD-STRATIFIED PAIRED STUDENT-CLUSTER BOOTSTRAP
# ============================================================

def build_fold_student_index(students, folds):
    """Map each outer fold -> student -> row indices."""
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    result = {}
    for fold in np.sort(np.unique(folds)):
        fold_mask = folds == fold
        fold_students = np.unique(students[fold_mask])

        if len(fold_students) == 0:
            raise ValueError(f"Outer fold {fold} contains no students.")

        result[int(fold)] = {
            student: np.flatnonzero(fold_mask & (students == student))
            for student in fold_students
        }

    return result


def paired_student_cluster_bootstrap(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
):
    """
    Fold-stratified paired cluster bootstrap.

    Within each outer fold, students are sampled with replacement. Whenever a
    student is sampled, all of that student's interaction rows are retained for
    BOTH models. A student sampled multiple times contributes the whole cluster
    multiple times.

    In every bootstrap replicate, the metric is calculated separately in each
    outer fold and then averaged equally across folds, matching the reported
    Average Score statistic.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    rng = np.random.default_rng(seed)
    fold_student_index = build_fold_student_index(students, folds)

    raw_differences = []
    advantages = []

    for _ in range(n_bootstrap):
        sampled_index_parts = []

        for _, student_map in fold_student_index.items():
            fold_students = np.asarray(list(student_map.keys()), dtype=object)

            sampled_students = rng.choice(
                fold_students,
                size=len(fold_students),
                replace=True,
            )

            sampled_index_parts.extend(
                student_map[student]
                for student in sampled_students
            )

        idx = np.concatenate(sampled_index_parts)

        fine_val = fold_average_metric(
            metric,
            y_true[idx],
            pred_fine[idx],
            folds[idx],
        )
        semantic_val = fold_average_metric(
            metric,
            y_true[idx],
            pred_semantic[idx],
            folds[idx],
        )

        if np.isnan(fine_val) or np.isnan(semantic_val):
            continue

        raw_differences.append(raw_difference(fine_val, semantic_val))
        advantages.append(fine_advantage(metric, fine_val, semantic_val))

    raw_differences = np.asarray(raw_differences, dtype=float)
    advantages = np.asarray(advantages, dtype=float)

    if len(advantages) == 0:
        return {
            "bootstrap_n": 0,
            "bootstrap_mean_raw_diff": np.nan,
            "raw_diff_ci_low": np.nan,
            "raw_diff_ci_high": np.nan,
            "bootstrap_mean_advantage_fine": np.nan,
            "advantage_ci_low": np.nan,
            "advantage_ci_high": np.nan,
        }

    return {
        "bootstrap_n": int(len(advantages)),
        "bootstrap_mean_raw_diff": float(np.mean(raw_differences)),
        "raw_diff_ci_low": float(np.percentile(raw_differences, 2.5)),
        "raw_diff_ci_high": float(np.percentile(raw_differences, 97.5)),
        "bootstrap_mean_advantage_fine": float(np.mean(advantages)),
        "advantage_ci_low": float(np.percentile(advantages, 2.5)),
        "advantage_ci_high": float(np.percentile(advantages, 97.5)),
    }


# ============================================================
# 5. PAIRED STUDENT-CLUSTER PERMUTATION TEST
# ============================================================

def paired_student_cluster_permutation_test(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_permutations=N_PERMUTATIONS,
    seed=RANDOM_SEED,
):
    """
    Two-sided paired permutation test at the STUDENT level.

    Under the null that the two configurations are exchangeable within a paired
    student cluster, each student's complete Fine/Semantic prediction vectors
    are swapped as one block with probability 0.5.

    The test statistic is the unweighted mean of the metric across outer folds.
    The same statistic is used for the observed data and every permutation.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    fine_obs, fine_sd, fine_fold_values = fold_metric_summary(
        metric, y_true, pred_fine, folds
    )
    semantic_obs, semantic_sd, semantic_fold_values = fold_metric_summary(
        metric, y_true, pred_semantic, folds
    )

    if np.isnan(fine_obs) or np.isnan(semantic_obs):
        raise ValueError(
            f"Observed fold-average {metric} could not be calculated."
        )

    observed_raw_diff = raw_difference(fine_obs, semantic_obs)
    observed_advantage = fine_advantage(metric, fine_obs, semantic_obs)

    unique_students, student_codes = np.unique(students, return_inverse=True)
    rng = np.random.default_rng(seed)

    extreme = 0
    valid = 0

    for _ in range(n_permutations):
        # One paired swap decision per student; all rows for that student move together.
        swap_by_student = rng.integers(
            0, 2, size=len(unique_students), dtype=np.int8
        )
        swap_rows = swap_by_student[student_codes].astype(bool)

        perm_fine = np.where(swap_rows, pred_semantic, pred_fine)
        perm_semantic = np.where(swap_rows, pred_fine, pred_semantic)

        fine_perm_val = fold_average_metric(
            metric, y_true, perm_fine, folds
        )
        semantic_perm_val = fold_average_metric(
            metric, y_true, perm_semantic, folds
        )

        if np.isnan(fine_perm_val) or np.isnan(semantic_perm_val):
            continue

        perm_advantage = fine_advantage(
            metric, fine_perm_val, semantic_perm_val
        )
        valid += 1

        # Two-sided test.
        if abs(perm_advantage) >= abs(observed_advantage) - 1e-15:
            extreme += 1

    # Monte Carlo permutation p-value with +1 correction.
    p_value = (extreme + 1) / (valid + 1) if valid > 0 else np.nan

    return {
        "fine_value": fine_obs,
        "fine_sd": fine_sd,
        "fine_fold_values": fine_fold_values,
        "semantic_value": semantic_obs,
        "semantic_sd": semantic_sd,
        "semantic_fold_values": semantic_fold_values,
        "raw_diff_fine_minus_semantic": observed_raw_diff,
        "advantage_fine": observed_advantage,
        "permutation_n": int(valid),
        "p_value": float(p_value) if not np.isnan(p_value) else np.nan,
    }


# ============================================================
# 6. RUN ALL SELECTED FIRST-ATTEMPT COMPARISONS
# ============================================================

def main():
    summary_rows = []

    print("=" * 110)
    print("FIRST-ATTEMPT SIGNIFICANCE: TOP FINE-GRAINED VS TOP SEMANTIC")
    print("Statistic: UNWEIGHTED MEAN OF OUTER-FOLD METRIC VALUES")
    print("=" * 110)
    print(f"OOF root: {OOF_ROOT.resolve()}")
    print(f"Threshold: p > {THRESHOLD}")
    print(f"Expected outer folds: {EXPECTED_N_OUTER_FOLDS}")
    print(f"Student-cluster bootstrap replicates: {N_BOOTSTRAP}")
    print(f"Student-cluster permutation replicates: {N_PERMUTATIONS}")

    for i, cfg in enumerate(COMPARISONS):
        metric = cfg["metric"]
        comparison_id = cfg["comparison_id"]

        print("\n" + "=" * 110)
        print(f"{comparison_id} | Metric: {metric}")
        print(f"Fine-grained: {cfg['fine_name']} [{cfg['fine_table_value']}]")
        print(f"Semantic:     {cfg['semantic_name']} [{cfg['semantic_table_value']}]")
        print(f"Fine file:    {cfg['fine_file']}")
        print(f"Semantic file:{cfg['semantic_file']}")

        (
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            diagnostics,
        ) = load_and_pair(cfg["fine_file"], cfg["semantic_file"])

        print(
            f"Paired rows: {diagnostics['Rows_Common']} | "
            f"Paired students: {diagnostics['Students_Common']} | "
            f"Outer folds: {diagnostics['Outer_Folds']} | "
            f"Fine coverage: {diagnostics['Fine_Coverage_in_Common']:.3%} | "
            f"Semantic coverage: {diagnostics['Semantic_Coverage_in_Common']:.3%}"
        )

        # Different deterministic seeds per comparison while preserving reproducibility.
        comparison_seed = RANDOM_SEED + i * 1000

        perm = paired_student_cluster_permutation_test(
            metric,
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            n_permutations=N_PERMUTATIONS,
            seed=comparison_seed,
        )

        # Verify that the OOF-derived fold mean ± sample SD reproduces the main table.
        verify_against_table(
            f"{comparison_id} Fine",
            perm["fine_value"],
            perm["fine_sd"],
            cfg["fine_table_value"],
        )
        verify_against_table(
            f"{comparison_id} Semantic",
            perm["semantic_value"],
            perm["semantic_sd"],
            cfg["semantic_table_value"],
        )

        boot = paired_student_cluster_bootstrap(
            metric,
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            n_bootstrap=N_BOOTSTRAP,
            seed=comparison_seed + 1,
        )

        print(
            "Fold-average Fine value:      "
            f"{perm['fine_value']:.6f} ± {perm['fine_sd']:.6f}"
        )
        print(
            "Fold-average Semantic value:  "
            f"{perm['semantic_value']:.6f} ± {perm['semantic_sd']:.6f}"
        )
        print(
            "Fine fold values:             "
            + ", ".join(f"{x:.6f}" for x in perm["fine_fold_values"])
        )
        print(
            "Semantic fold values:         "
            + ", ".join(f"{x:.6f}" for x in perm["semantic_fold_values"])
        )
        print(
            "Fine - Semantic raw diff:     "
            f"{perm['raw_diff_fine_minus_semantic']:.6f}"
        )
        print(
            "Fine advantage (positive=fine better): "
            f"{perm['advantage_fine']:.6f}"
        )
        print(
            "95% bootstrap CI for raw Fine-Semantic difference: "
            f"[{boot['raw_diff_ci_low']:.6f}, {boot['raw_diff_ci_high']:.6f}]"
        )
        print(
            "95% bootstrap CI for Fine advantage: "
            f"[{boot['advantage_ci_low']:.6f}, {boot['advantage_ci_high']:.6f}]"
        )
        print(f"Student-cluster permutation p: {perm['p_value']:.6g}")

        summary_rows.append({
            "Comparison_ID": comparison_id,
            "Metric": metric,
            "Fine_Model": cfg["fine_name"],
            "Fine_Table_MeanSD": cfg["fine_table_value"],
            "Semantic_Model": cfg["semantic_name"],
            "Semantic_Table_MeanSD": cfg["semantic_table_value"],
            "Fine_File": str(cfg["fine_file"]),
            "Semantic_File": str(cfg["semantic_file"]),
            "Rows_Fine": diagnostics["Rows_Fine"],
            "Rows_Semantic": diagnostics["Rows_Semantic"],
            "Rows_Common": diagnostics["Rows_Common"],
            "Students_Common": diagnostics["Students_Common"],
            "Outer_Folds": diagnostics["Outer_Folds"],
            "Fine_Coverage_in_Common": diagnostics["Fine_Coverage_in_Common"],
            "Semantic_Coverage_in_Common": diagnostics["Semantic_Coverage_in_Common"],
            "Fold_Average_OOF_Fine_Value": perm["fine_value"],
            "Fold_SD_OOF_Fine_Value": perm["fine_sd"],
            "Fold_Average_OOF_Semantic_Value": perm["semantic_value"],
            "Fold_SD_OOF_Semantic_Value": perm["semantic_sd"],
            "Raw_Diff_Fine_minus_Semantic": perm["raw_diff_fine_minus_semantic"],
            "Advantage_Fine_PositiveMeansFineBetter": perm["advantage_fine"],
            "Bootstrap_Mean_Raw_Diff": boot["bootstrap_mean_raw_diff"],
            "Bootstrap_RawDiff_CI95_Low": boot["raw_diff_ci_low"],
            "Bootstrap_RawDiff_CI95_High": boot["raw_diff_ci_high"],
            "Bootstrap_Mean_Advantage_Fine": boot["bootstrap_mean_advantage_fine"],
            "Bootstrap_Advantage_CI95_Low": boot["advantage_ci_low"],
            "Bootstrap_Advantage_CI95_High": boot["advantage_ci_high"],
            "Permutation_P_Value": perm["p_value"],
            "Permutation_N": perm["permutation_n"],
            "Bootstrap_N": boot["bootstrap_n"],
        })

    # ============================================================
    # 7. MULTIPLE-COMPARISON CORRECTION + SAVE
    # ============================================================

    summary_df = pd.DataFrame(summary_rows)

    valid_p_mask = summary_df["Permutation_P_Value"].notna()
    summary_df["Holm_P_Value"] = np.nan
    summary_df["Significant_Holm_0.05"] = False

    if valid_p_mask.any():
        rejected, p_holm, _, _ = multipletests(
            summary_df.loc[valid_p_mask, "Permutation_P_Value"].to_numpy(),
            alpha=ALPHA,
            method="holm",
        )
        summary_df.loc[valid_p_mask, "Holm_P_Value"] = p_holm
        summary_df.loc[valid_p_mask, "Significant_Holm_0.05"] = rejected

    # Descriptive indicators only. Formal significance is based on
    # Holm-adjusted permutation p-values.
    summary_df["RawDiff_CI_Excludes_Zero"] = (
        (summary_df["Bootstrap_RawDiff_CI95_Low"] > 0)
        | (summary_df["Bootstrap_RawDiff_CI95_High"] < 0)
    )
    summary_df["Advantage_CI_Excludes_Zero"] = (
        (summary_df["Bootstrap_Advantage_CI95_Low"] > 0)
        | (summary_df["Bootstrap_Advantage_CI95_High"] < 0)
    )

    output_file = "first_attempt_fine_vs_semantic_significance.csv"
    summary_df.to_csv(output_file, index=False)

    print("\n" + "=" * 110)
    print("FINAL SUMMARY")
    print("=" * 110)

    cols_to_print = [
        "Comparison_ID",
        "Metric",
        "Fine_Model",
        "Semantic_Model",
        "Rows_Common",
        "Students_Common",
        "Fold_Average_OOF_Fine_Value",
        "Fold_Average_OOF_Semantic_Value",
        "Raw_Diff_Fine_minus_Semantic",
        "Advantage_Fine_PositiveMeansFineBetter",
        "Bootstrap_RawDiff_CI95_Low",
        "Bootstrap_RawDiff_CI95_High",
        "Permutation_P_Value",
        "Holm_P_Value",
        "Significant_Holm_0.05",
    ]

    with pd.option_context(
        "display.max_columns", None,
        "display.width", 260,
        "display.max_colwidth", 55,
    ):
        print(summary_df[cols_to_print].to_string(index=False))

    print(f"\nSaved summary to: {output_file}")


if __name__ == "__main__":
    main()

In [ ]:
# After Feedback Averaged Statistical Significance 

"""
After Feedback statistical significance analysis:
best fine-grained vs best semantic DLKT configuration for each metric.

STATISTIC USED
--------------
The reported performance statistic is the UNWEIGHTED MEAN of the metric
calculated separately in each outer test fold. This matches the "Average Score"
reported in the main results table.

Inference is still performed from paired out-of-fold predictions while respecting
student-level dependence:

  1) two-sided paired STUDENT-CLUSTER permutation test for p-values;
  2) fold-stratified paired STUDENT-CLUSTER bootstrap for 95% confidence intervals;
  3) Holm correction across the seven metric-wise tests within this prediction stage.

For every observed/permuted/bootstrap dataset, the statistic is computed in the
same way:

    metric in fold 1
    metric in fold 2
    metric in fold 3
        -> unweighted mean across outer folds

Students remain intact during permutation/bootstrap. Fine and semantic predictions
are paired on exactly the same row_id values, and student_id, y_true, and fold must
agree between the two OOF files.

Expected OOF layout
-------------------
    AKT_AF/oof_AKT_<kc_name>_all_with_student_id.csv
    SAKT_AF/oof_SAKT_<kc_name>_all.csv
    DKVMN_AF/oof_DKVMN_<kc_name>_all.csv
    SKVMN_AF/oof_SKVMN_<kc_name>_all.csv
    DKT_AF/oof_DKT_<kc_name>_all.csv
    DKTForget_AF/oof_DKTForget_<kc_name>_all.csv

Required OOF columns:
    row_id, student_id, fold, y_true, y_pred, model_name, kc_name
"""

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
)
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Run this script from the directory containing AKT_AF, DKTForget_AF, etc.,
# or change OOF_ROOT to the parent directory containing those folders.
OOF_ROOT = Path(".")

# Match the KT wrappers exactly: probability must be STRICTLY greater than 0.5.
THRESHOLD = 0.5

# Your outer CV uses three folds. This is checked in every paired comparison.
EXPECTED_N_OUTER_FOLDS = 3

# Student-cluster resampling settings.
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 10000
RANDOM_SEED = 42
ALPHA = 0.05

# Number of decimals used by the displayed main results table.
TABLE_DECIMALS = 3

# Exercise ID is treated as fine-grained.
FINE_GRAINED_KCS = {
    "actionableelementid",
    "itemid",
    "itemsetid",
    "exerciseid",
}

SEMANTIC_KCS = {
    "propertyexercisetype",
    "linguisticconstructs",
    "propertyid",
}


def oof_path(folder: str, model: str, kc: str) -> Path:
    """Build OOF path, using the student-ID-fixed AKT files."""
    if model == "AKT":
        return OOF_ROOT / folder / f"oof_{model}_{kc}_all_with_student_id.csv"
    return OOF_ROOT / folder / f"oof_{model}_{kc}_all.csv"


# Best fine-grained and semantic configuration for each metric,
# selected from the AFTER-FEEDBACK Average Score table.
# Values below are displayed to three decimals only.
#
# NOTE ON MAE:
# The unrounded fold-average values resolve the rounded tie between the
# fine-grained candidates:
#   DKT + Item Set ID = 0.195803 ± 0.002680
#   AKT + Exercise ID = 0.195633 ± 0.007185
# Lower MAE is better, so AKT + Exercise ID is retained as the winner.
COMPARISONS = [
    {
        "comparison_id": "AUC",
        "metric": "AUC",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.871 ± 0.012",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.851 ± 0.009",
    },
    {
        "comparison_id": "Accuracy",
        "metric": "Accuracy",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.856 ± 0.006",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.833 ± 0.006",
    },
    {
        "comparison_id": "RMSE",
        "metric": "RMSE",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.336 ± 0.009",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.353 ± 0.007",
    },
    {
        "comparison_id": "MAE",
        "metric": "MAE",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.196 ± 0.007",
        "semantic_name": "DKT+Forget + Property + Exercise Type",
        "semantic_file": oof_path(
            "DKTForget_AF", "DKTForget", "propertyexercisetype"
        ),
        "semantic_table_value": "0.216 ± 0.007",
    },
    {
        "comparison_id": "Precision",
        "metric": "Precision",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.880 ± 0.008",
        "semantic_name": "AKT + Linguistic Constructs",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "linguisticconstructs"
        ),
        "semantic_table_value": "0.857 ± 0.003",
    },
    {
        "comparison_id": "Recall",
        "metric": "Recall",
        "fine_name": "DKVMN + Exercise ID",
        "fine_file": oof_path(
            "DKVMN_AF", "DKVMN", "exerciseid"
        ),
        "fine_table_value": "0.940 ± 0.007",
        "semantic_name": "SKVMN + Property ID",
        "semantic_file": oof_path(
            "SKVMN_AF", "SKVMN", "propertyid"
        ),
        "semantic_table_value": "0.935 ± 0.009",
    },
    {
        "comparison_id": "F1",
        "metric": "F1",
        "fine_name": "AKT + Exercise ID",
        "fine_file": oof_path(
            "AKT_AF", "AKT", "exerciseid"
        ),
        "fine_table_value": "0.905 ± 0.003",
        "semantic_name": "AKT + Property + Exercise Type",
        "semantic_file": oof_path(
            "AKT_AF", "AKT", "propertyexercisetype"
        ),
        "semantic_table_value": "0.892 ± 0.004",
    },
]

if len(COMPARISONS) != 7:
    raise RuntimeError(
        f"Expected exactly 7 comparisons, got {len(COMPARISONS)}."
    )


# ============================================================
# 2. METRICS + FOLD-AVERAGED STATISTIC
# ============================================================

def rmse_score(y_true, y_prob) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_prob) ** 2)))


def metric_value(metric: str, y_true, y_prob) -> float:
    """Calculate one metric on one set of interaction-level predictions."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(y_true) == 0:
        return np.nan

    if metric == "AUC":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))

    # Match model wrappers exactly: p > 0.5 -> 1, otherwise 0.
    y_hat = (y_prob > THRESHOLD).astype(int)

    if metric == "Accuracy":
        return float(accuracy_score(y_true, y_hat))
    if metric == "RMSE":
        return rmse_score(y_true, y_prob)
    if metric == "MAE":
        return float(mean_absolute_error(y_true, y_prob))
    if metric == "Precision":
        return float(precision_score(y_true, y_hat, zero_division=0))
    if metric == "Recall":
        return float(recall_score(y_true, y_hat, zero_division=0))
    if metric == "F1":
        return float(f1_score(y_true, y_hat, zero_division=0))

    raise ValueError(f"Unsupported metric: {metric}")


def fold_metric_values(metric: str, y_true, y_prob, folds) -> np.ndarray:
    """
    Calculate the requested metric separately in every outer fold.

    The returned values are the same type of fold-level scores used to build
    the Average Score (mean ± sample SD) in the main results table.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    folds = np.asarray(folds, dtype=int)

    if not (len(y_true) == len(y_prob) == len(folds)):
        raise ValueError("y_true, y_prob, and folds must have the same length.")

    unique_folds = np.sort(np.unique(folds))
    if len(unique_folds) == 0:
        return np.asarray([], dtype=float)

    values = []
    for fold in unique_folds:
        mask = folds == fold
        value = metric_value(metric, y_true[mask], y_prob[mask])
        if np.isnan(value):
            return np.asarray([], dtype=float)
        values.append(value)

    return np.asarray(values, dtype=float)


def fold_metric_summary(metric: str, y_true, y_prob, folds):
    """
    Return the unweighted mean and sample SD of the outer-fold metric values.

    Sample SD uses ddof=1, matching the SD convention used in the supplied
    after-feedback Average Score table.
    """
    values = fold_metric_values(metric, y_true, y_prob, folds)

    if len(values) == 0:
        return np.nan, np.nan, values

    mean_value = float(np.mean(values))
    sd_value = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
    return mean_value, sd_value, values


def fold_average_metric(metric: str, y_true, y_prob, folds) -> float:
    """Unweighted mean of the requested metric across outer folds."""
    mean_value, _, _ = fold_metric_summary(metric, y_true, y_prob, folds)
    return mean_value


def raw_difference(fine_value: float, semantic_value: float) -> float:
    """Raw Model A - Model B difference."""
    return float(fine_value - semantic_value)


def fine_advantage(metric: str, fine_value: float, semantic_value: float) -> float:
    """
    Direction-standardized difference.

    Positive always means the fine-grained configuration is better.
      - Higher-is-better metrics: fine - semantic
      - Lower-is-better metrics (RMSE/MAE): semantic - fine
    """
    if metric in {"RMSE", "MAE"}:
        return float(semantic_value - fine_value)
    return float(fine_value - semantic_value)


def parse_table_mean_sd(value: str):
    """Parse a string such as '0.825 ± 0.003'."""
    parts = value.split("±")
    if len(parts) != 2:
        raise ValueError(f"Could not parse table value: {value!r}")
    return float(parts[0].strip()), float(parts[1].strip())


def verify_against_table(
    label: str,
    computed_mean: float,
    computed_sd: float,
    table_value: str,
    decimals: int = TABLE_DECIMALS,
):
    """
    Verify that OOF-derived fold mean and sample SD reproduce the displayed
    Average Score after rounding to the table precision.

    This is a safeguard only; the hard-coded table strings are never used as
    inputs to the significance calculations.
    """
    table_mean, table_sd = parse_table_mean_sd(table_value)

    mean_matches = f"{computed_mean:.{decimals}f}" == f"{table_mean:.{decimals}f}"
    sd_matches = f"{computed_sd:.{decimals}f}" == f"{table_sd:.{decimals}f}"

    if not (mean_matches and sd_matches):
        raise ValueError(
            f"{label}: OOF-derived fold summary does not match the supplied "
            f"Average Score after rounding to {decimals} decimals.\n"
            f"Computed: {computed_mean:.6f} ± {computed_sd:.6f}\n"
            f"Table:    {table_value}\n"
            "Check that the correct OOF files/folds are being used."
        )


# ============================================================
# 3. LOAD + STRICTLY PAIR OOF ROWS
# ============================================================

REQUIRED_OOF_COLUMNS = {
    "row_id",
    "student_id",
    "fold",
    "y_true",
    "y_pred",
    "model_name",
    "kc_name",
}


def read_oof(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing OOF file:\n  {path}\n"
            "Check OOF_ROOT and make sure the expected *_all.csv file exists."
        )

    df = pd.read_csv(path)

    missing = REQUIRED_OOF_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"{path} is missing required OOF columns: {sorted(missing)}.\n"
            "Required columns are row_id, student_id, fold, y_true, y_pred, "
            "model_name, and kc_name."
        )

    if df["row_id"].duplicated().any():
        n_dup = int(df["row_id"].duplicated().sum())
        raise ValueError(f"{path} contains {n_dup} duplicate row_id values.")

    analysis_cols = ["row_id", "student_id", "fold", "y_true", "y_pred"]
    if df[analysis_cols].isna().any().any():
        raise ValueError(
            f"{path} contains missing values in required analysis columns."
        )

    # Every student must belong to exactly one fixed outer fold.
    per_student_fold_count = df.groupby("student_id")["fold"].nunique()
    if (per_student_fold_count > 1).any():
        bad = per_student_fold_count[per_student_fold_count > 1].index.tolist()[:10]
        raise ValueError(
            f"{path}: some students appear in multiple outer folds. Examples: {bad}"
        )

    n_folds = int(df["fold"].nunique())
    if n_folds != EXPECTED_N_OUTER_FOLDS:
        raise ValueError(
            f"{path}: expected {EXPECTED_N_OUTER_FOLDS} outer folds, "
            f"but found {n_folds}: {sorted(df['fold'].unique().tolist())}"
        )

    return df


def load_and_pair(file_fine: Path, file_semantic: Path):
    fine = read_oof(file_fine)
    semantic = read_oof(file_semantic)

    n_fine = len(fine)
    n_semantic = len(semantic)

    fine_keep = fine[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    sem_keep = semantic[
        ["row_id", "student_id", "fold", "y_true", "y_pred"]
    ].copy()

    # --------------------------------------------------------
    # STRICT ROW-COVERAGE CHECK
    # --------------------------------------------------------
    # Both configurations must contain exactly the same held-out interactions.
    fine_row_ids = set(fine_keep["row_id"])
    semantic_row_ids = set(sem_keep["row_id"])

    missing_from_semantic = fine_row_ids - semantic_row_ids
    missing_from_fine = semantic_row_ids - fine_row_ids

    if missing_from_semantic or missing_from_fine:
        raise ValueError(
            "OOF row coverage does not match exactly between the two configurations.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Rows missing from Semantic: {len(missing_from_semantic)}\n"
            f"Rows missing from Fine: {len(missing_from_fine)}\n"
            "The paired significance test requires identical held-out "
            "interaction rows."
        )

    # --------------------------------------------------------
    # PAIR EXACTLY ON row_id
    # --------------------------------------------------------
    paired = fine_keep.merge(
        sem_keep,
        on="row_id",
        how="inner",
        suffixes=("_fine", "_semantic"),
        validate="one_to_one",
    ).sort_values("row_id").reset_index(drop=True)

    if len(paired) != n_fine or len(paired) != n_semantic:
        raise ValueError(
            "Unexpected row-count mismatch after pairing.\n"
            f"Fine rows: {n_fine}\n"
            f"Semantic rows: {n_semantic}\n"
            f"Paired rows: {len(paired)}"
        )

    # --------------------------------------------------------
    # VERIFY y_true, student_id, AND fold MATCH
    # --------------------------------------------------------
    fine_y = paired["y_true_fine"].astype(int).to_numpy()
    semantic_y = paired["y_true_semantic"].astype(int).to_numpy()
    if not np.array_equal(fine_y, semantic_y):
        raise ValueError(
            "y_true mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    fine_students = paired["student_id_fine"].astype(str).to_numpy()
    semantic_students = paired["student_id_semantic"].astype(str).to_numpy()
    if not np.array_equal(fine_students, semantic_students):
        raise ValueError(
            "student_id mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    fine_folds = paired["fold_fine"].astype(int).to_numpy()
    semantic_folds = paired["fold_semantic"].astype(int).to_numpy()
    if not np.array_equal(fine_folds, semantic_folds):
        raise ValueError(
            "Outer-fold mismatch on paired row_id values between:\n"
            f"  {file_fine}\n"
            f"  {file_semantic}"
        )

    y_true = fine_y
    pred_fine = paired["y_pred_fine"].astype(float).to_numpy()
    pred_semantic = paired["y_pred_semantic"].astype(float).to_numpy()
    students = fine_students
    folds = fine_folds

    diagnostics = {
        "Rows_Fine": n_fine,
        "Rows_Semantic": n_semantic,
        "Rows_Common": len(paired),
        "Students_Common": int(pd.Series(students).nunique()),
        "Outer_Folds": int(pd.Series(folds).nunique()),
        "Fine_Coverage_in_Common": len(paired) / n_fine,
        "Semantic_Coverage_in_Common": len(paired) / n_semantic,
    }

    return (
        y_true,
        pred_fine,
        pred_semantic,
        students,
        folds,
        diagnostics,
    )


# ============================================================
# 4. FOLD-STRATIFIED PAIRED STUDENT-CLUSTER BOOTSTRAP
# ============================================================

def build_fold_student_index(students, folds):
    """Map each outer fold -> student -> row indices."""
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    result = {}
    for fold in np.sort(np.unique(folds)):
        fold_mask = folds == fold
        fold_students = np.unique(students[fold_mask])

        if len(fold_students) == 0:
            raise ValueError(f"Outer fold {fold} contains no students.")

        result[int(fold)] = {
            student: np.flatnonzero(fold_mask & (students == student))
            for student in fold_students
        }

    return result


def paired_student_cluster_bootstrap(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
):
    """
    Fold-stratified paired cluster bootstrap.

    Within each outer fold, students are sampled with replacement. Whenever a
    student is sampled, all of that student's interaction rows are retained for
    BOTH models. A student sampled multiple times contributes the whole cluster
    multiple times.

    In every bootstrap replicate, the metric is calculated separately in each
    outer fold and then averaged equally across folds, matching the reported
    Average Score statistic.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    rng = np.random.default_rng(seed)
    fold_student_index = build_fold_student_index(students, folds)

    raw_differences = []
    advantages = []

    for _ in range(n_bootstrap):
        sampled_index_parts = []

        for _, student_map in fold_student_index.items():
            fold_students = np.asarray(list(student_map.keys()), dtype=object)

            sampled_students = rng.choice(
                fold_students,
                size=len(fold_students),
                replace=True,
            )

            sampled_index_parts.extend(
                student_map[student]
                for student in sampled_students
            )

        idx = np.concatenate(sampled_index_parts)

        fine_val = fold_average_metric(
            metric,
            y_true[idx],
            pred_fine[idx],
            folds[idx],
        )
        semantic_val = fold_average_metric(
            metric,
            y_true[idx],
            pred_semantic[idx],
            folds[idx],
        )

        if np.isnan(fine_val) or np.isnan(semantic_val):
            continue

        raw_differences.append(raw_difference(fine_val, semantic_val))
        advantages.append(fine_advantage(metric, fine_val, semantic_val))

    raw_differences = np.asarray(raw_differences, dtype=float)
    advantages = np.asarray(advantages, dtype=float)

    if len(advantages) == 0:
        return {
            "bootstrap_n": 0,
            "bootstrap_mean_raw_diff": np.nan,
            "raw_diff_ci_low": np.nan,
            "raw_diff_ci_high": np.nan,
            "bootstrap_mean_advantage_fine": np.nan,
            "advantage_ci_low": np.nan,
            "advantage_ci_high": np.nan,
        }

    return {
        "bootstrap_n": int(len(advantages)),
        "bootstrap_mean_raw_diff": float(np.mean(raw_differences)),
        "raw_diff_ci_low": float(np.percentile(raw_differences, 2.5)),
        "raw_diff_ci_high": float(np.percentile(raw_differences, 97.5)),
        "bootstrap_mean_advantage_fine": float(np.mean(advantages)),
        "advantage_ci_low": float(np.percentile(advantages, 2.5)),
        "advantage_ci_high": float(np.percentile(advantages, 97.5)),
    }


# ============================================================
# 5. PAIRED STUDENT-CLUSTER PERMUTATION TEST
# ============================================================

def paired_student_cluster_permutation_test(
    metric,
    y_true,
    pred_fine,
    pred_semantic,
    students,
    folds,
    n_permutations=N_PERMUTATIONS,
    seed=RANDOM_SEED,
):
    """
    Two-sided paired permutation test at the STUDENT level.

    Under the null that the two configurations are exchangeable within a paired
    student cluster, each student's complete Fine/Semantic prediction vectors
    are swapped as one block with probability 0.5.

    The test statistic is the unweighted mean of the metric across outer folds.
    The same statistic is used for the observed data and every permutation.
    """
    y_true = np.asarray(y_true)
    pred_fine = np.asarray(pred_fine)
    pred_semantic = np.asarray(pred_semantic)
    students = np.asarray(students)
    folds = np.asarray(folds, dtype=int)

    fine_obs, fine_sd, fine_fold_values = fold_metric_summary(
        metric, y_true, pred_fine, folds
    )
    semantic_obs, semantic_sd, semantic_fold_values = fold_metric_summary(
        metric, y_true, pred_semantic, folds
    )

    if np.isnan(fine_obs) or np.isnan(semantic_obs):
        raise ValueError(
            f"Observed fold-average {metric} could not be calculated."
        )

    observed_raw_diff = raw_difference(fine_obs, semantic_obs)
    observed_advantage = fine_advantage(metric, fine_obs, semantic_obs)

    unique_students, student_codes = np.unique(students, return_inverse=True)
    rng = np.random.default_rng(seed)

    extreme = 0
    valid = 0

    for _ in range(n_permutations):
        # One paired swap decision per student; all rows for that student move together.
        swap_by_student = rng.integers(
            0, 2, size=len(unique_students), dtype=np.int8
        )
        swap_rows = swap_by_student[student_codes].astype(bool)

        perm_fine = np.where(swap_rows, pred_semantic, pred_fine)
        perm_semantic = np.where(swap_rows, pred_fine, pred_semantic)

        fine_perm_val = fold_average_metric(
            metric, y_true, perm_fine, folds
        )
        semantic_perm_val = fold_average_metric(
            metric, y_true, perm_semantic, folds
        )

        if np.isnan(fine_perm_val) or np.isnan(semantic_perm_val):
            continue

        perm_advantage = fine_advantage(
            metric, fine_perm_val, semantic_perm_val
        )
        valid += 1

        # Two-sided test.
        if abs(perm_advantage) >= abs(observed_advantage) - 1e-15:
            extreme += 1

    # Monte Carlo permutation p-value with +1 correction.
    p_value = (extreme + 1) / (valid + 1) if valid > 0 else np.nan

    return {
        "fine_value": fine_obs,
        "fine_sd": fine_sd,
        "fine_fold_values": fine_fold_values,
        "semantic_value": semantic_obs,
        "semantic_sd": semantic_sd,
        "semantic_fold_values": semantic_fold_values,
        "raw_diff_fine_minus_semantic": observed_raw_diff,
        "advantage_fine": observed_advantage,
        "permutation_n": int(valid),
        "p_value": float(p_value) if not np.isnan(p_value) else np.nan,
    }


# ============================================================
# 6. RUN ALL SELECTED AFTER-FEEDBACK COMPARISONS
# ============================================================

def main():
    summary_rows = []

    print("=" * 110)
    print("AFTER-FEEDBACK SIGNIFICANCE: TOP FINE-GRAINED VS TOP SEMANTIC")
    print("Statistic: UNWEIGHTED MEAN OF OUTER-FOLD METRIC VALUES")
    print("=" * 110)
    print(f"OOF root: {OOF_ROOT.resolve()}")
    print(f"Threshold: p > {THRESHOLD}")
    print(f"Expected outer folds: {EXPECTED_N_OUTER_FOLDS}")
    print(f"Student-cluster bootstrap replicates: {N_BOOTSTRAP}")
    print(f"Student-cluster permutation replicates: {N_PERMUTATIONS}")

    for i, cfg in enumerate(COMPARISONS):
        metric = cfg["metric"]
        comparison_id = cfg["comparison_id"]

        print("\n" + "=" * 110)
        print(f"{comparison_id} | Metric: {metric}")
        print(f"Fine-grained: {cfg['fine_name']} [{cfg['fine_table_value']}]")
        print(f"Semantic:     {cfg['semantic_name']} [{cfg['semantic_table_value']}]")
        print(f"Fine file:    {cfg['fine_file']}")
        print(f"Semantic file:{cfg['semantic_file']}")

        (
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            diagnostics,
        ) = load_and_pair(cfg["fine_file"], cfg["semantic_file"])

        print(
            f"Paired rows: {diagnostics['Rows_Common']} | "
            f"Paired students: {diagnostics['Students_Common']} | "
            f"Outer folds: {diagnostics['Outer_Folds']} | "
            f"Fine coverage: {diagnostics['Fine_Coverage_in_Common']:.3%} | "
            f"Semantic coverage: {diagnostics['Semantic_Coverage_in_Common']:.3%}"
        )

        # Different deterministic seeds per comparison while preserving reproducibility.
        comparison_seed = RANDOM_SEED + i * 1000

        perm = paired_student_cluster_permutation_test(
            metric,
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            n_permutations=N_PERMUTATIONS,
            seed=comparison_seed,
        )

        # Verify that the OOF-derived fold mean ± sample SD reproduces the main table.
        verify_against_table(
            f"{comparison_id} Fine",
            perm["fine_value"],
            perm["fine_sd"],
            cfg["fine_table_value"],
        )
        verify_against_table(
            f"{comparison_id} Semantic",
            perm["semantic_value"],
            perm["semantic_sd"],
            cfg["semantic_table_value"],
        )

        boot = paired_student_cluster_bootstrap(
            metric,
            y_true,
            pred_fine,
            pred_semantic,
            students,
            folds,
            n_bootstrap=N_BOOTSTRAP,
            seed=comparison_seed + 1,
        )

        print(
            "Fold-average Fine value:      "
            f"{perm['fine_value']:.6f} ± {perm['fine_sd']:.6f}"
        )
        print(
            "Fold-average Semantic value:  "
            f"{perm['semantic_value']:.6f} ± {perm['semantic_sd']:.6f}"
        )
        print(
            "Fine fold values:             "
            + ", ".join(f"{x:.6f}" for x in perm["fine_fold_values"])
        )
        print(
            "Semantic fold values:         "
            + ", ".join(f"{x:.6f}" for x in perm["semantic_fold_values"])
        )
        print(
            "Fine - Semantic raw diff:     "
            f"{perm['raw_diff_fine_minus_semantic']:.6f}"
        )
        print(
            "Fine advantage (positive=fine better): "
            f"{perm['advantage_fine']:.6f}"
        )
        print(
            "95% bootstrap CI for raw Fine-Semantic difference: "
            f"[{boot['raw_diff_ci_low']:.6f}, {boot['raw_diff_ci_high']:.6f}]"
        )
        print(
            "95% bootstrap CI for Fine advantage: "
            f"[{boot['advantage_ci_low']:.6f}, {boot['advantage_ci_high']:.6f}]"
        )
        print(f"Student-cluster permutation p: {perm['p_value']:.6g}")

        summary_rows.append({
            "Comparison_ID": comparison_id,
            "Metric": metric,
            "Fine_Model": cfg["fine_name"],
            "Fine_Table_MeanSD": cfg["fine_table_value"],
            "Semantic_Model": cfg["semantic_name"],
            "Semantic_Table_MeanSD": cfg["semantic_table_value"],
            "Fine_File": str(cfg["fine_file"]),
            "Semantic_File": str(cfg["semantic_file"]),
            "Rows_Fine": diagnostics["Rows_Fine"],
            "Rows_Semantic": diagnostics["Rows_Semantic"],
            "Rows_Common": diagnostics["Rows_Common"],
            "Students_Common": diagnostics["Students_Common"],
            "Outer_Folds": diagnostics["Outer_Folds"],
            "Fine_Coverage_in_Common": diagnostics["Fine_Coverage_in_Common"],
            "Semantic_Coverage_in_Common": diagnostics["Semantic_Coverage_in_Common"],
            "Fold_Average_OOF_Fine_Value": perm["fine_value"],
            "Fold_SD_OOF_Fine_Value": perm["fine_sd"],
            "Fold_Average_OOF_Semantic_Value": perm["semantic_value"],
            "Fold_SD_OOF_Semantic_Value": perm["semantic_sd"],
            "Raw_Diff_Fine_minus_Semantic": perm["raw_diff_fine_minus_semantic"],
            "Advantage_Fine_PositiveMeansFineBetter": perm["advantage_fine"],
            "Bootstrap_Mean_Raw_Diff": boot["bootstrap_mean_raw_diff"],
            "Bootstrap_RawDiff_CI95_Low": boot["raw_diff_ci_low"],
            "Bootstrap_RawDiff_CI95_High": boot["raw_diff_ci_high"],
            "Bootstrap_Mean_Advantage_Fine": boot["bootstrap_mean_advantage_fine"],
            "Bootstrap_Advantage_CI95_Low": boot["advantage_ci_low"],
            "Bootstrap_Advantage_CI95_High": boot["advantage_ci_high"],
            "Permutation_P_Value": perm["p_value"],
            "Permutation_N": perm["permutation_n"],
            "Bootstrap_N": boot["bootstrap_n"],
        })

    # ============================================================
    # 7. MULTIPLE-COMPARISON CORRECTION + SAVE
    # ============================================================

    summary_df = pd.DataFrame(summary_rows)

    valid_p_mask = summary_df["Permutation_P_Value"].notna()
    summary_df["Holm_P_Value"] = np.nan
    summary_df["Significant_Holm_0.05"] = False

    if valid_p_mask.any():
        rejected, p_holm, _, _ = multipletests(
            summary_df.loc[valid_p_mask, "Permutation_P_Value"].to_numpy(),
            alpha=ALPHA,
            method="holm",
        )
        summary_df.loc[valid_p_mask, "Holm_P_Value"] = p_holm
        summary_df.loc[valid_p_mask, "Significant_Holm_0.05"] = rejected

    # Descriptive indicators only. Formal significance is based on
    # Holm-adjusted permutation p-values.
    summary_df["RawDiff_CI_Excludes_Zero"] = (
        (summary_df["Bootstrap_RawDiff_CI95_Low"] > 0)
        | (summary_df["Bootstrap_RawDiff_CI95_High"] < 0)
    )
    summary_df["Advantage_CI_Excludes_Zero"] = (
        (summary_df["Bootstrap_Advantage_CI95_Low"] > 0)
        | (summary_df["Bootstrap_Advantage_CI95_High"] < 0)
    )

    output_file = "after_feedback_fine_vs_semantic_significance.csv"
    summary_df.to_csv(output_file, index=False)

    print("\n" + "=" * 110)
    print("FINAL SUMMARY")
    print("=" * 110)

    cols_to_print = [
        "Comparison_ID",
        "Metric",
        "Fine_Model",
        "Semantic_Model",
        "Rows_Common",
        "Students_Common",
        "Fold_Average_OOF_Fine_Value",
        "Fold_Average_OOF_Semantic_Value",
        "Raw_Diff_Fine_minus_Semantic",
        "Advantage_Fine_PositiveMeansFineBetter",
        "Bootstrap_RawDiff_CI95_Low",
        "Bootstrap_RawDiff_CI95_High",
        "Permutation_P_Value",
        "Holm_P_Value",
        "Significant_Holm_0.05",
    ]

    with pd.option_context(
        "display.max_columns", None,
        "display.width", 260,
        "display.max_colwidth", 55,
    ):
        print(summary_df[cols_to_print].to_string(index=False))

    print(f"\nSaved summary to: {output_file}")


if __name__ == "__main__":
    main()